In [2]:
import psycopg2, pandas as pd

def get_db_conn():
    return psycopg2.connect(
        host='194.171.191.227',
        port=5432,
        dbname='o2',
        user='postgres',
        password='postgres',
        connect_timeout=10
    )

LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik',
    'Maurice of Oranje-Nassau': 'Maurits',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem van Oranje',
    'Money of necessity minted during the siege of breda': 'Noodmunten',
    'City map of Breda in relief': 'Stadsplan in relief',
    'Model of a 24-pound gun': 'kanon',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht van de Spaanse bezetting',
    'Military Operations in flanders with siege and capture of breda': 'beleg van Breda',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht van de Spaanse bezetting',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
}

print('Ready')

Ready


In [4]:
import psycopg2

conn = get_db_conn()
cur = conn.cursor()

results = []
for cv_label, db_term in LABEL_TO_DB.items():
    # Try exact title match
    cur.execute("SELECT COUNT(*) FROM knowledge_base WHERE title ILIKE %s", (f'%{db_term}%',))
    count = cur.fetchone()[0]
    results.append({
        'cv_label': cv_label,
        'db_search_term': db_term,
        'title_matches': count
    })

cur.close()
conn.close()

df_debug = pd.DataFrame(results)
print(df_debug.to_string())

                                                             cv_label                      db_search_term  title_matches
0                                                  Justinus of Nassau                 Justinus van Nassau              4
1                                    Frederick Henry of Orange-Nassau                    Frederik Hendrik             39
2                                            Maurice of Oranje-Nassau                             Maurits             62
3                              Equestrian statue of William of Orange  ruiterstandbeeld Willem van Oranje              0
4                 Money of necessity minted during the siege of breda                          Noodmunten              0
5                                         City map of Breda in relief                 Stadsplan in relief              0
6                                             Model of a 24-pound gun                               kanon             10
7       Constantijn Huygens, cur

In [5]:
conn = get_db_conn()
cur = conn.cursor()

# Get ALL BR% artifacts
cur.execute("SELECT identifier, title FROM knowledge_base WHERE identifier LIKE 'BR%' ORDER BY identifier")
rows = cur.fetchall()

print(f'All {len(rows)} BR% museum artifacts:\n')
for r in rows:
    print(f'  {r[0]} -> {r[1]}')

cur.close()
conn.close()

All 31 BR% museum artifacts:

  BR00003 -> Justinus van Nassau
  BR00011 -> Portret Willem van Oranje
  BR00012 -> Maurits, prins van Oranje-Nassau
  BR00013 -> Portret van Frederik Hendrik van Oranje Nassau
  BR00014 -> Drinkglas in de vorm van een kanon
  BR00015 -> Wijnfles
  BR00019 -> Tazza
  BR00022 -> Fragment van Maria met Christuskind
  BR00023 -> Fragment van kleding van vrouw
  BR00024 -> Fragment van Voetwassing en Laatste Avondmaal: beeld met Maria Magdalena
  BR00025 -> Portret van Ludovicus Requesens
  BR00026 -> Portret van Willem I van Nassau (van Oranje)
  BR00027 -> Portret van Philips II van Spanje
  BR00028 -> Portret van Ferdinand Alvares van Toledo
  BR00029 -> Laatste Avondmaal uit de Grote Kerk
  BR00030 -> Vernield beeld uit de Grote Kerk
  BR00031 -> Mogelijk fragment van de Voetwassing uit de Grote Kerk
  BR00032 -> Filips Willem, prins van Oranje
  BR00038 -> Stadsplan in reliëf van de stad Breda
  BR00039 -> Stilleven met aronskelken
  BR00040 -> Observato

In [9]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

pkl_labels = [
    'Albrecht van Oostenrijk', 'Ambrogio Spinola', 'Beeld vna de turfschipper',
    'City map of Breda in relief', 'Constantijn Huygens, curator van de Illustere School in Breda',
    'Dataset Begijnhof Statue', 'Dataset Begijntjes Statue', 'Dataset Dancing Lamb',
    'Dataset Museum Gate', 'Dataset Museum Wall', 'Dataset Tobias en de Engel Statue',
    'Dataset Watch Tower Remains', 'Departure of the Spanish occupation from Breda on 10 October 1637',
    'Documents', 'Equestrian statue of William of Orange', 'Filips Willem, prins van Oranje',
    'Frederick Henry of Orange-Nassau', 'Frederik Hendrik, Prins van Oranje',
    'Herdenkingsbeker van het beleg van Breda in 1637', 'Het overlijden van Prins Maurits',
    'Isabella Clara Eugenia van Spanja', 'Justinus of Nassau', 'Maurice of Oranje-Nassau',
    'Military Operations in flanders with siege and capture of breda', 'Model of a 24-pound gun',
    'Model van het Turfschip', 'Money of necessity minted during the siege of breda',
    'Noodmunten', 'ONZEKER', 'Obsidio Bredana',
    'Portret van Jan de Jongere, graaf van nassau Siegen',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen',
    'Retreat of the Spanish Garrison from Breda', 'Schelpdieren en vis', 'Sporen van de Nassaus',
    'Turfschip van Breda in 1590', 'collectie 17de en 18de eeuws metaal militair tuig',
    'de overgave van breda', 'het beleg van Breda', 'partizaan, 17de eeuw, metaal',
    'portret van lodewijk van Nassau'
]

for label in pkl_labels:
    words = [w for w in label.split() if len(w) > 3][:2]
    if not words:
        print(f'{label[:55]:<55} → ❌ too short')
        continue
    search = '%' + '%'.join(words) + '%'
    cur.execute("SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s AND identifier LIKE 'BR%%' LIMIT 1", (search,))
    rows = cur.fetchall()
    if rows:
        print(f'{label[:55]:<55} → {rows[0][0]} | {rows[0][1]}')
    else:
        print(f'{label[:55]:<55} → ❌ NO MATCH')

cur.close()
conn.close()

Albrecht van Oostenrijk                                 → ❌ NO MATCH
Ambrogio Spinola                                        → ❌ NO MATCH
Beeld vna de turfschipper                               → ❌ NO MATCH
City map of Breda in relief                             → ❌ NO MATCH
Constantijn Huygens, curator van de Illustere School in → ❌ NO MATCH
Dataset Begijnhof Statue                                → ❌ NO MATCH
Dataset Begijntjes Statue                               → ❌ NO MATCH
Dataset Dancing Lamb                                    → ❌ NO MATCH
Dataset Museum Gate                                     → ❌ NO MATCH
Dataset Museum Wall                                     → ❌ NO MATCH
Dataset Tobias en de Engel Statue                       → ❌ NO MATCH
Dataset Watch Tower Remains                             → ❌ NO MATCH
Departure of the Spanish occupation from Breda on 10 Oc → ❌ NO MATCH
Documents                                               → ❌ NO MATCH
Equestrian statue of William of Or

In [10]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Manual correct mapping based on what we know is in the DB
correct_mappings = {
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',  # search non-BR records
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker',
    'Justinus of Nassau': 'Justinus van Nassau',
    'het beleg van Breda': 'Herdenkingsbeker',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik',
    'Maurice of Oranje-Nassau': 'Maurits',
    'Isabella Clara Eugenia van Spanja': 'Isabella',
    'portret van lodewijk van Nassau': 'Lodewijk',
}

for label, search_term in correct_mappings.items():
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s LIMIT 3",
        (f'%{search_term}%',)
    )
    rows = cur.fetchall()
    print(f'\n{label}:')
    for r in rows:
        print(f'  {r[0]} | {r[1]}')

cur.close()
conn.close()


Albrecht van Oostenrijk:
  ST02224 | Albrecht van Oostenrijk

Ambrogio Spinola:
  ST00629 | Ambrogio Spinola
  ST02230 | Portret van Ambrogio Spinola
  ST00588 | Ambrogio Spinola

Filips Willem, prins van Oranje:
  ST04906 | Filips Willem van Nassau
  BR00032 | Filips Willem, prins van Oranje
  ST04907 | Filips Willem Prins van Oranje

Herdenkingsbeker van het beleg van Breda in 1637:
  BR00048 | Herdenkingsbeker n.a.v. het beleg van Breda in 1637

Justinus of Nassau:
  ST04854 | Justinus van Nassau
  BR00003 | Justinus van Nassau
  ST04852 | Justinus van Nassau

het beleg van Breda:
  BR00048 | Herdenkingsbeker n.a.v. het beleg van Breda in 1637

Frederik Hendrik, Prins van Oranje:
  ST05213 | Frederik Hendrik van Nassau
  ST05215 | Frederik Hendrik, prins van Oranje
  ST05217 | Frederik Hendrik van Nassau

Frederick Henry of Orange-Nassau:
  ST05213 | Frederik Hendrik van Nassau
  ST05215 | Frederik Hendrik, prins van Oranje
  ST05217 | Frederik Hendrik van Nassau

Maurice of Oranje

In [11]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

pkl_labels = [
    'Albrecht van Oostenrijk', 'Ambrogio Spinola', 'Beeld vna de turfschipper',
    'City map of Breda in relief', 'Constantijn Huygens, curator van de Illustere School in Breda',
    'Dataset Begijnhof Statue', 'Dataset Begijntjes Statue', 'Dataset Dancing Lamb',
    'Dataset Museum Gate', 'Dataset Museum Wall', 'Dataset Tobias en de Engel Statue',
    'Dataset Watch Tower Remains', 'Departure of the Spanish occupation from Breda on 10 October 1637',
    'Documents', 'Equestrian statue of William of Orange', 'Filips Willem, prins van Oranje',
    'Frederick Henry of Orange-Nassau', 'Frederik Hendrik, Prins van Oranje',
    'Herdenkingsbeker van het beleg van Breda in 1637', 'Het overlijden van Prins Maurits',
    'Isabella Clara Eugenia van Spanja', 'Justinus of Nassau', 'Maurice of Oranje-Nassau',
    'Military Operations in flanders with siege and capture of breda', 'Model of a 24-pound gun',
    'Model van het Turfschip', 'Money of necessity minted during the siege of breda',
    'Noodmunten', 'ONZEKER', 'Obsidio Bredana',
    'Portret van Jan de Jongere, graaf van nassau Siegen',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen',
    'Retreat of the Spanish Garrison from Breda', 'Schelpdieren en vis', 'Sporen van de Nassaus',
    'Turfschip van Breda in 1590', 'collectie 17de en 18de eeuws metaal militair tuig',
    'de overgave van breda', 'het beleg van Breda', 'partizaan, 17de eeuw, metaal',
    'portret van lodewijk van Nassau'
]

for label in pkl_labels:
    words = [w for w in label.split() if len(w) > 3][:1]
    if not words:
        print(f'{label[:55]:<55} → ❌ too short')
        continue
    cur.execute(
        "SELECT COUNT(*) FROM knowledge_base WHERE title ILIKE %s",
        (f'%{words[0]}%',)
    )
    count = cur.fetchone()[0]
    status = '✅' if count > 0 else '❌'
    print(f'{status} {label[:55]:<55} → {count} records')

cur.close()
conn.close()

✅ Albrecht van Oostenrijk                                 → 1 records
✅ Ambrogio Spinola                                        → 3 records
✅ Beeld vna de turfschipper                               → 1528 records
✅ City map of Breda in relief                             → 8 records
✅ Constantijn Huygens, curator van de Illustere School in → 4 records
❌ Dataset Begijnhof Statue                                → 0 records
❌ Dataset Begijntjes Statue                               → 0 records
❌ Dataset Dancing Lamb                                    → 0 records
❌ Dataset Museum Gate                                     → 0 records
❌ Dataset Museum Wall                                     → 0 records
❌ Dataset Tobias en de Engel Statue                       → 0 records
❌ Dataset Watch Tower Remains                             → 0 records
❌ Departure of the Spanish occupation from Breda on 10 Oc → 0 records
❌ Documents                                               → 0 records
❌ Equestrian stat

In [12]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

pkl_labels = [
    'Albrecht van Oostenrijk', 'Ambrogio Spinola', 'Beeld vna de turfschipper',
    'City map of Breda in relief', 'Constantijn Huygens, curator van de Illustere School in Breda',
    'Dataset Begijnhof Statue', 'Dataset Begijntjes Statue', 'Dataset Dancing Lamb',
    'Dataset Museum Gate', 'Dataset Museum Wall', 'Dataset Tobias en de Engel Statue',
    'Dataset Watch Tower Remains', 'Departure of the Spanish occupation from Breda on 10 October 1637',
    'Documents', 'Equestrian statue of William of Orange', 'Filips Willem, prins van Oranje',
    'Frederick Henry of Orange-Nassau', 'Frederik Hendrik, Prins van Oranje',
    'Herdenkingsbeker van het beleg van Breda in 1637', 'Het overlijden van Prins Maurits',
    'Isabella Clara Eugenia van Spanja', 'Justinus of Nassau', 'Maurice of Oranje-Nassau',
    'Military Operations in flanders with siege and capture of breda', 'Model of a 24-pound gun',
    'Model van het Turfschip', 'Money of necessity minted during the siege of breda',
    'Noodmunten', 'ONZEKER', 'Obsidio Bredana',
    'Portret van Jan de Jongere, graaf van nassau Siegen',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen',
    'Retreat of the Spanish Garrison from Breda', 'Schelpdieren en vis', 'Sporen van de Nassaus',
    'Turfschip van Breda in 1590', 'collectie 17de en 18de eeuws metaal militair tuig',
    'de overgave van breda', 'het beleg van Breda', 'partizaan, 17de eeuw, metaal',
    'portret van lodewijk van Nassau'
]

for label in pkl_labels:
    # Search the label directly
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s LIMIT 1",
        (f'%{label[:20]}%',)
    )
    rows = cur.fetchall()
    if rows:
        print(f'✅ {label[:50]:<50} → {rows[0][0]} | {rows[0][1][:50]}')
    else:
        print(f'❌ {label[:50]:<50} → NO MATCH')

cur.close()
conn.close()

✅ Albrecht van Oostenrijk                            → ST02224 | Albrecht van Oostenrijk
✅ Ambrogio Spinola                                   → ST00629 | Ambrogio Spinola
❌ Beeld vna de turfschipper                          → NO MATCH
❌ City map of Breda in relief                        → NO MATCH
✅ Constantijn Huygens, curator van de Illustere Scho → 0003009 | Postzegelvel, Nederland 75 cent, De wereld gaet ha
❌ Dataset Begijnhof Statue                           → NO MATCH
❌ Dataset Begijntjes Statue                          → NO MATCH
❌ Dataset Dancing Lamb                               → NO MATCH
❌ Dataset Museum Gate                                → NO MATCH
❌ Dataset Museum Wall                                → NO MATCH
❌ Dataset Tobias en de Engel Statue                  → NO MATCH
❌ Dataset Watch Tower Remains                        → NO MATCH
❌ Departure of the Spanish occupation from Breda on  → NO MATCH
❌ Documents                                          → NO MATCH
❌ Equestr

In [13]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

check_labels = {
    'Departure of the Spanish occupation from Breda on 10 October 1637': ['Uittocht', 'Spaanse bezetting', 'oktober 1637'],
    'Retreat of the Spanish Garrison from Breda': ['Uittocht', 'Spaanse garnizoen', 'Terugtocht'],
    'Military Operations in flanders with siege and capture of breda': ['Militaire operaties', 'beleg', 'Vlaanderen'],
    'Equestrian statue of William of Orange': ['ruiterstandbeeld', 'Willem van Oranje', 'ruiter'],
    'Model of a 24-pound gun': ['24-pond', 'kanon', 'geschut', 'vuurwapen'],
    'partizaan, 17de eeuw, metaal': ['partizaan', 'spies', 'lans', 'hellebaard'],
}

for label, search_terms in check_labels.items():
    print(f'\n{label}:')
    for term in search_terms:
        cur.execute(
            "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s LIMIT 2",
            (f'%{term}%',)
        )
        rows = cur.fetchall()
        if rows:
            for r in rows:
                print(f'  ✅ "{term}" → {r[0]} | {r[1]}')
        else:
            print(f'  ❌ "{term}" → no match')

cur.close()
conn.close()


Departure of the Spanish occupation from Breda on 10 October 1637:
  ✅ "Uittocht" → ST00072 | Uittocht van het Spaanse garnizoen uit Breda, 1637
  ✅ "Uittocht" → ST06931 | Uittocht van de Spaanse bezetting
  ✅ "Spaanse bezetting" → ST06931 | Uittocht van de Spaanse bezetting
  ✅ "Spaanse bezetting" → S01073 | De uittocht der Spaanse bezetting uit Breda op 10 oktober 1637
  ✅ "oktober 1637" → S01073 | De uittocht der Spaanse bezetting uit Breda op 10 oktober 1637

Retreat of the Spanish Garrison from Breda:
  ✅ "Uittocht" → S01073 | De uittocht der Spaanse bezetting uit Breda op 10 oktober 1637
  ✅ "Uittocht" → ST00072 | Uittocht van het Spaanse garnizoen uit Breda, 1637
  ✅ "Spaanse garnizoen" → ST00072 | Uittocht van het Spaanse garnizoen uit Breda, 1637
  ❌ "Terugtocht" → no match

Military Operations in flanders with siege and capture of breda:
  ❌ "Militaire operaties" → no match
  ✅ "beleg" → ST00065 | Spotprent op de financiering van het beleg van Breda door Spinola
  ✅ "beleg" 

In [14]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Full mapping: cv_label → best search term
full_mapping = {
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Beeld vna de turfschipper': 'turfschipper',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Dataset Begijnhof Statue': 'Begijnhof',
    'Dataset Begijntjes Statue': 'Begijntjes',
    'Dataset Dancing Lamb': 'dansend lam',
    'Dataset Museum Gate': 'museumpoort',
    'Dataset Museum Wall': 'museummuur',
    'Dataset Tobias en de Engel Statue': 'Tobias Engel',
    'Dataset Watch Tower Remains': 'wachttoren',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht Spaanse bezetting',
    'Documents': 'document',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker',
    'Het overlijden van Prins Maurits': 'overlijden Maurits',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Justinus of Nassau': 'Justinus van Nassau',
    'Maurice of Oranje-Nassau': 'Maurits',
    'Military Operations in flanders with siege and capture of breda': 'beleg Breda Spinola',
    'Model of a 24-pound gun': 'kanon',
    'Model van het Turfschip': 'Turfschip',
    'Money of necessity minted during the siege of breda': 'noodmunt',
    'Noodmunten': 'noodmunt',
    'ONZEKER': 'onbekend',
    'Obsidio Bredana': 'Obsidio Bredana',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Jan van Nassau',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen': 'meselaarsgilde',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht garnizoen',
    'Schelpdieren en vis': 'Schelpdieren',
    'Sporen van de Nassaus': 'Sporen Nassau',
    'Turfschip van Breda in 1590': 'Turfschip Breda',
    'collectie 17de en 18de eeuws metaal militair tuig': 'militair tuig',
    'de overgave van breda': 'overgave Breda',
    'het beleg van Breda': 'beleg Breda',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'portret van lodewijk van Nassau': 'Lodewijk Nassau',
}

print(f'Checking {len(full_mapping)} labels...\n')
found = 0
not_found = []

for label, search_term in full_mapping.items():
    words = search_term.split()
    search = f'%{words[0]}%'
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s LIMIT 1",
        (search,)
    )
    rows = cur.fetchall()
    if rows:
        found += 1
        print(f'✅ {label[:50]:<50} → {rows[0][0]} | {rows[0][1][:40]}')
    else:
        not_found.append(label)
        print(f'❌ {label[:50]:<50} → NO MATCH')

cur.close()
conn.close()

print(f'\n--- SUMMARY ---')
print(f'Found: {found}/{len(full_mapping)}')
print(f'Not found: {len(not_found)}')
for l in not_found:
    print(f'  ❌ {l}')

Checking 42 labels...

✅ Albrecht van Oostenrijk                            → ST02224 | Albrecht van Oostenrijk
✅ Ambrogio Spinola                                   → ST00629 | Ambrogio Spinola
✅ Beeld vna de turfschipper                          → S00274 | Legpenning onthulling standbeeld Turfsch
✅ City map of Breda in relief                        → BR00038 | Stadsplan in reliëf van de stad Breda
✅ Constantijn Huygens, curator van de Illustere Scho → ST00571 | Constantijn Huijgens
✅ Dataset Begijnhof Statue                           → ST01835 | Breda, Begijnhof
✅ Dataset Begijntjes Statue                          → GT01282 | Begijnhof, drie begijntjes bij de waterp
✅ Dataset Dancing Lamb                               → G00149 | Beeld van dansende Faun of Satyr
❌ Dataset Museum Gate                                → NO MATCH
❌ Dataset Museum Wall                                → NO MATCH
✅ Dataset Tobias en de Engel Statue                  → BT01666 | Genezing van Tobits blindheid, Tob

In [15]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

verify = {
    'Turfschip van Breda in 1590': ['Turfschip van Breda', 'Turfschip Breda 1590'],
    'Sporen van de Nassaus': ['Nassaus Breda', 'Sporen Nassau', 'Nassau Breda'],
    'collectie 17de en 18de eeuws metaal militair tuig': ['militair tuig', 'metaal militair', '17de eeuws'],
    'Portret van Jan de Jongere, graaf van nassau Siegen': ['Jan van Nassau', 'Nassau Siegen', 'Jan de Jongere'],
    'portret van lodewijk van Nassau': ['Lodewijk van Nassau', 'Lodewijk Nassau'],
    'Model van het Turfschip': ['Turfschip', 'model turfschip'],
    'Schelpdieren en vis': ['Schelpdieren vis', 'Schelpdieren Breda'],
}

for label, terms in verify.items():
    print(f'\n{label}:')
    for term in terms:
        cur.execute(
            "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s LIMIT 2",
            (f'%{term}%',)
        )
        rows = cur.fetchall()
        if rows:
            for r in rows:
                print(f'  ✅ "{term}" → {r[0]} | {r[1]}')
        else:
            print(f'  ❌ "{term}" → no match')

cur.close()
conn.close()


Turfschip van Breda in 1590:
  ✅ "Turfschip van Breda" → ST00027 | Turfschip van Breda in 1590
  ❌ "Turfschip Breda 1590" → no match

Sporen van de Nassaus:
  ❌ "Nassaus Breda" → no match
  ❌ "Sporen Nassau" → no match
  ❌ "Nassau Breda" → no match

collectie 17de en 18de eeuws metaal militair tuig:
  ❌ "militair tuig" → no match
  ❌ "metaal militair" → no match
  ❌ "17de eeuws" → no match

Portret van Jan de Jongere, graaf van nassau Siegen:
  ❌ "Jan van Nassau" → no match
  ❌ "Nassau Siegen" → no match
  ❌ "Jan de Jongere" → no match

portret van lodewijk van Nassau:
  ✅ "Lodewijk van Nassau" → ST05081 | Willem Lodewijk van Nassau
  ✅ "Lodewijk van Nassau" → ST05082 | Willem Lodewijk van Nassau
  ❌ "Lodewijk Nassau" → no match

Model van het Turfschip:
  ✅ "Turfschip" → 0003599 | kaart 25e jazz festival breda live at the turfschip
  ✅ "Turfschip" → G04314 | ‘t Turfschip
  ❌ "model turfschip" → no match

Schelpdieren en vis:
  ❌ "Schelpdieren vis" → no match
  ❌ "Schelpdieren Breda" 

In [17]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

verify2 = {
    'Sporen van de Nassaus': ['Sporen', 'Nassau', 'Nassaus'],
    'collectie 17de en 18de eeuws metaal militair tuig': ['collectie metaal', 'militaire collectie', 'metaal tuig', 'collectie wapen', 'wapentuig'],
    'Portret van Jan de Jongere, graaf van nassau Siegen': ['graaf Nassau', 'Nassau Siegen', 'Jan Nassau', 'Portret Jan'],
    'Model van het Turfschip': ['Model Turfschip', 'Turfschip model', 'miniatuur turfschip'],
    'Schelpdieren en vis': ['Schelpdieren', 'schelp', 'vis Breda'],
}

for label, terms in verify2.items():
    print(f'\n{label}:')
    for term in terms:
        cur.execute(
            "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s LIMIT 2",
            (f'%{term}%',)
        )
        rows = cur.fetchall()
        if rows:
            for r in rows:
                print(f'  ✅ "{term}" → {r[0]} | {r[1]}')
        else:
            print(f'  ❌ "{term}" → no match')

cur.close()
conn.close()


Sporen van de Nassaus:
  ✅ "Sporen" → GT01239 | Deel van de vatenhal, waar de houten fusten met een inhoud van 16 tot 106 liter worden ‘afgedrukt’ om lekkage op te sporen
  ✅ "Sporen" → 0004137 | Rosbeek 57, het huis van San Claessens, in de voetsporen van De Stijl en Het Nieuwe Bouwen.
  ✅ "Nassau" → BR00012 | Maurits, prins van Oranje-Nassau
  ✅ "Nassau" → BR00013 | Portret van Frederik Hendrik van Oranje Nassau
  ✅ "Nassaus" → GT04845 | Nassaustraat
  ✅ "Nassaus" → GT04413 | Nieuwe Boschstraat en Nassaustraat

collectie 17de en 18de eeuws metaal militair tuig:
  ❌ "collectie metaal" → no match
  ❌ "militaire collectie" → no match
  ❌ "metaal tuig" → no match
  ❌ "collectie wapen" → no match
  ❌ "wapentuig" → no match

Portret van Jan de Jongere, graaf van nassau Siegen:
  ❌ "graaf Nassau" → no match
  ❌ "Nassau Siegen" → no match
  ❌ "Jan Nassau" → no match
  ✅ "Portret Jan" → SMB000047 | Zelfportret Jan Theuns
  ✅ "Portret Jan" → ST02306 | Portret Janus Wagemakers

Model van het T

In [3]:
! pip install opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---- ----------------------------------- 4.7/40.2 MB 27.3 MB/s eta 0:00:02
   ----------------- ---------------------- 17.8/40.2 MB 46.0 MB/s eta 0:00:01
   ------------------------------------- -- 37.7/40.2 MB 62.7 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 57.9 MB/s  0:00:00


In [5]:
import requests, json, base64
import cv2
from pathlib import Path

# Extract frame
video_path = r'C:\Users\jimal\OneDrive\Documents\UGUIDU\Database Operations\DatasetsAlexi\Albrecht van Oostenrijk\20251212_123531.mp4'
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

print(f'Frame read: {ret}')
print(f'Frame shape: {frame.shape if ret else "N/A"}')

# Encode as JPEG
_, buf = cv2.imencode('.jpg', frame)
img_bytes = buf.tobytes()
img_b64 = base64.b64encode(img_bytes).decode('utf-8')

print(f'Base64 length: {len(img_b64)}')
print(f'First 50 chars: {img_b64[:50]}')

# Check what classify endpoint expects
resp = requests.post(
    'http://194.171.191.226:3414/api/classify',
    json={'image': img_b64},
    timeout=30
)
print(f'\nStatus: {resp.status_code}')
print(f'Response: {resp.json()}')

Frame read: True
Frame shape: (1920, 1080, 3)
Base64 length: 468664
First 50 chars: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAIBAQEBAQIBAQECAg

Status: 422
Response: {'detail': [{'type': 'missing', 'loc': ['body', 'image'], 'msg': 'Field required', 'input': None, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]}


In [24]:
import requests, json, base64, os
import cv2
from pathlib import Path

BACKEND = 'http://194.171.191.226:3414'

LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Nassau',
    'Maurice of Oranje-Nassau': 'Maurits van Nassau',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem',
    'Money of necessity minted during the siege of breda': 'Noodmunt Breda',
    'Noodmunten': 'Noodmunt Breda',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'Model of a 24-pound gun': 'kanon Breda',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht Spaanse bezetting',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht Spaanse garnizoen',
    'Military Operations in flanders with siege and capture of breda': 'beleg van Breda Spinola',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker beleg Breda',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'Turfschip van Breda',
    'Turfschip van Breda in 1590': 'Turfschip van Breda',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'overgave van Breda',
    'collectie 17de en 18de eeuws metaal militair tuig': 'militair tuig',
    'Sporen van de Nassaus': 'Sporen Nassaus',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Jan van Nassau',
    'portret van lodewijk van Nassau': 'Lodewijk van Nassau',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Het overlijden van Prins Maurits': 'overlijden Prins Maurits',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik van Nassau',
}

def classify_image(img_bytes):
    resp = requests.post(
        f'{BACKEND}/api/classify',
        files={'image': ('frame.jpg', img_bytes, 'image/jpeg')},
        timeout=30
    )
    data = resp.json()
    return {
        'label': data.get('landmark', 'unknown'),
        'confidence': data.get('confidence', 0),
        'status': data.get('status', 'unknown')
    }

def get_rag_response(label):
    parts = []
    with requests.post(
        f'{BACKEND}/api/chat',
        json={'message': label, 'landmark': label, 'conversation_id': 'notebook-test', 'audience': 'tourist'},
        stream=True, timeout=60
    ) as resp:
        for line in resp.iter_lines():
            if line:
                try:
                    d = json.loads(line.decode('utf-8') if isinstance(line, bytes) else line)
                    if d.get('type') == 'answer':
                        parts.append(d.get('content', ''))
                except:
                    pass
    return ''.join(parts).strip()

def test_artifact(source):
    """Test with URL, local image path, or local video path."""
    print(f'Source: {source}')

    if source.startswith('http'):
        print('Fetching from URL...')
        img_bytes = requests.get(source, timeout=10).content
        r = classify_image(img_bytes)
        best_label = r.get('label', 'unknown')
        best_conf = r.get('confidence', 0)
        best_status = r.get('status', '')

    elif source.lower().endswith(('.mp4', '.avi', '.mov')):
        print('Extracting best frame from video...')
        cap = cv2.VideoCapture(source)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f'Total frames: {total_frames} | FPS: {fps:.1f}')

        best_label, best_conf, best_status = 'unknown', 0, ''

        for sec in [0, 1, 2, 3, 4]:
            pos = int(fps * sec)
            if pos >= total_frames:
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
            ret, frame = cap.read()
            if not ret:
                continue
            _, buf = cv2.imencode('.jpg', frame)
            img_bytes = buf.tobytes()
            r = classify_image(img_bytes)
            lbl = r.get('label', 'unknown')
            conf = r.get('confidence', 0)
            status = r.get('status', '')
            print(f'  {sec}s (frame {pos}): {lbl} ({conf:.3f}) [{status}]')
            if lbl != 'unknown' and conf > best_conf:
                best_conf = conf
                best_label = lbl
                best_status = status

        cap.release()

    else:
        print('Reading local image...')
        img_bytes = Path(source).read_bytes()
        r = classify_image(img_bytes)
        best_label = r.get('label', 'unknown')
        best_conf = r.get('confidence', 0)
        best_status = r.get('status', '')

    print(f'\n--- CV Classification ---')
    print(f'Label:      {best_label}')
    print(f'Confidence: {best_conf:.3f}')
    print(f'Status:     {best_status}')
    print(f'DB term:    {LABEL_TO_DB.get(best_label, best_label)}')

    if best_label == 'unknown' or best_conf < 0.3:
        print('Low confidence or unknown — skipping RAG')
        return

    print('\n--- RAG Response ---')
    response = get_rag_response(best_label)
    print(response)

# ── Test here ─────────────────────────────────────────────────
test_artifact(r"C:\Users\jimal\OneDrive\Documents\UGUIDU\Database Operations\Backup Dataset- 12.12.2025\collectie 17de en 18de eeuws metaal militair tuig\20251212_122432.mp4")

Source: C:\Users\jimal\OneDrive\Documents\UGUIDU\Database Operations\Backup Dataset- 12.12.2025\collectie 17de en 18de eeuws metaal militair tuig\20251212_122432.mp4
Extracting best frame from video...
Total frames: 483 | FPS: 30.0
  0s (frame 0): Filips Willem, prins van Oranje (0.841) [UNCERTAIN]
  1s (frame 30): Filips Willem, prins van Oranje (0.812) [UNCERTAIN]
  2s (frame 60): collectie 17de en 18de eeuws metaal militair tuig (0.827) [UNCERTAIN]
  3s (frame 90): collectie 17de en 18de eeuws metaal militair tuig (0.777) [UNCERTAIN]
  4s (frame 120): collectie 17de en 18de eeuws metaal militair tuig (0.836) [UNCERTAIN]

--- CV Classification ---
Label:      Filips Willem, prins van Oranje
Confidence: 0.841
Status:     UNCERTAIN
DB term:    Filips Willem

--- RAG Response ---
Filips Willem was the only son of Willem van Oranje and Anna van Egmont. He was named after Filips II of Spain. He spent his youth in Breda and Leuven. He restored Breda's dignity after returning in 1609. He di

In [25]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

cur.execute("""
    SELECT identifier, title, object_type, description
    FROM knowledge_base 
    WHERE title ILIKE '%halfharnas%'
    OR title ILIKE '%rapier%'
    OR title ILIKE '%bourgondisch%'
    OR title ILIKE '%stormhoed%'
    OR title ILIKE '%piekenier%'
    OR description ILIKE '%halfharnas%'
    OR description ILIKE '%piekenier%'
    LIMIT 10
""")
for r in cur.fetchall():
    print(f'{r[0]} | {r[1]} | {r[2]}')
    print(f'  {r[3][:150] if r[3] else "NULL"}')
    print()

cur.close()
conn.close()

S01496 | Borstharnas | Halfharnas
  Dit betreft een halfharnas van een piekenier. Deze droeg een 5.5 meter lange piek en een helm. Gebruikt in de periode 1600-1675 en geproduceerd in de 

S07193 | Rapier | Rapier
  17de-eeuwse rapier. Normaliter zijn de gevesten niet van koper\\\\messing. Gevest is verguld en eindigt in vijf versierde eikels.

G00845 | Bourgondisch landschap / landschap met bloemen | Schilderij
  Schilderij, olieverf op doek met Bourgondisch landschap. Abstracte voorstelling, niet gesigneerd in houten lijst.

S00095 | Rapier | Rapier
  Degen met cilindrische degenknop met aan einde sierknop.

S01492 | Spaans rapier | Rapier
  Lang tweesnijdend lemmet met gegraveerde beugel. De rechte armen van het degenkruis zijn door een boog verbonden, de degenknop is afgeplat en de spil 

S01495 | Bourgondische stormhoed | Helm
  Het betreft een Bourgondische stormhoed uit de periode 1550-1600. Zeer waarschijnlijk in Duitsland geproduceerd. Het bekken in twee stukken met hoge k

S01

In [26]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Test all new search terms
test_queries = {
    'collectie 17de en 18de eeuws metaal militair tuig': 'halfharnas piekenier rapier',
    'Model of a 24-pound gun': 'saluutkanon wapen Breda',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'Het overlijden van Prins Maurits': 'overlijden Prins Maurits',
    'Register van het meselaarsgilde': 'Furie Houtepen',
    'Portret van Jan de Jongere': 'Jan van Nassau-Siegen',
    'portret van lodewijk van Nassau': 'Willem Lodewijk van Nassau',
    'de overgave van breda': 'Las Lanzas',
    'Turfschip van Breda in 1590': 'Turfschip van Breda',
}

for label, search_term in test_queries.items():
    words = search_term.split()
    
    # Simulate the ILIKE fallback with multiple words
    conditions = ' OR '.join([f"title ILIKE %s OR content ILIKE %s" for _ in words])
    params = []
    for w in words:
        params.extend([f'%{w}%', f'%{w}%'])
    
    cur.execute(f"""
        SELECT identifier, title, object_type
        FROM knowledge_base
        WHERE {conditions}
        ORDER BY
            CASE WHEN identifier LIKE 'BR%%' THEN 0
                 WHEN identifier LIKE 'S%%' THEN 1
                 ELSE 2 END
        LIMIT 3
    """, params)
    
    rows = cur.fetchall()
    status = '✅' if rows else '❌'
    print(f'\n{status} {label[:50]}')
    print(f'   Search: "{search_term}"')
    for r in rows:
        print(f'   → {r[0]} | {r[1]} | {r[2]}')

cur.close()
conn.close()


✅ collectie 17de en 18de eeuws metaal militair tuig
   Search: "halfharnas piekenier rapier"
   → S01496 | Borstharnas | Halfharnas
   → S07193 | Rapier | Rapier
   → S01493 | Degen à la Pappenheim | Rapier

✅ Model of a 24-pound gun
   Search: "saluutkanon wapen Breda"
   → BR00030 | Vernield beeld uit de Grote Kerk | Beeldhouwwerk
   → BR00031 | Mogelijk fragment van de Voetwassing uit de Grote Kerk | Beeldhouwwerk
   → BR00045.1-3 | Vrede van Munster 1648-1998 | C-print

✅ City map of Breda in relief
   Search: "Stadsplan reliëf"
   → BR00022 | Fragment van Maria met Christuskind | Beeldhouwwerk
   → BR00030 | Vernield beeld uit de Grote Kerk | Beeldhouwwerk
   → BR00031 | Mogelijk fragment van de Voetwassing uit de Grote Kerk | Beeldhouwwerk

✅ Het overlijden van Prins Maurits
   Search: "overlijden Prins Maurits"
   → BR00012 | Maurits, prins van Oranje-Nassau | Schilderij
   → BR00013 | Portret van Frederik Hendrik van Oranje Nassau | Schilderij
   → BR00003 | Justinus van Nassa

In [27]:
# Simulate exactly how llm_service.py does the ILIKE search
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

test_queries = {
    'Model of a 24-pound gun': 'saluutkanon wapen Breda',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'de overgave van breda': 'Las Lanzas',
    'collectie 17de en 18de eeuws metaal militair tuig': 'halfharnas piekenier rapier',
    'Register van het meselaarsgilde': 'Furie Houtepen',
    'Portret van Jan de Jongere': 'Jan van Nassau-Siegen',
}

for label, search_term in test_queries.items():
    # Single ILIKE as llm_service does it
    cur.execute("""
        SELECT identifier, title, object_type
        FROM knowledge_base
        WHERE title ILIKE %s OR content ILIKE %s
        LIMIT 3
    """, (f'%{search_term}%', f'%{search_term}%'))
    
    rows = cur.fetchall()
    status = '✅' if rows else '❌'
    print(f'\n{status} {label[:50]}')
    print(f'   Search: "{search_term}"')
    for r in rows:
        print(f'   → {r[0]} | {r[1]} | {r[2]}')

cur.close()
conn.close()


❌ Model of a 24-pound gun
   Search: "saluutkanon wapen Breda"

❌ City map of Breda in relief
   Search: "Stadsplan reliëf"

✅ de overgave van breda
   Search: "Las Lanzas"
   → G00456 | De overgave van Breda (Las Lanzas) | Schilderij
   → GT02329 | Schilderij “ Las Lanzas” in de hal van stadhuis | Glasnegatief
   → GT02355 | Schilderij “ Las Lanzas” in de hal van stadhuis | Glasnegatief

❌ collectie 17de en 18de eeuws metaal militair tuig
   Search: "halfharnas piekenier rapier"

❌ Register van het meselaarsgilde
   Search: "Furie Houtepen"

❌ Portret van Jan de Jongere
   Search: "Jan van Nassau-Siegen"


In [29]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Better specific single terms
test_queries = {
    'Turfschip van Breda in 1590': 'Turfschip Breda',
    'Het overlijden van Prins Maurits': 'Maurits gestorven',
    'Register van het meselaarsgilde': 'Furie van de Houtepen',
    'portret van lodewijk van Nassau': 'Willem Lodewijk Nassau',
}

for label, search_term in test_queries.items():
    # Try each word separately with AND logic
    words = search_term.split()
    if len(words) == 1:
        cur.execute("""
            SELECT identifier, title, object_type
            FROM knowledge_base
            WHERE title ILIKE %s OR content ILIKE %s
            LIMIT 3
        """, (f'%{words[0]}%', f'%{words[0]}%'))
    else:
        # Use first two words as separate conditions
        cur.execute("""
            SELECT identifier, title, object_type
            FROM knowledge_base
            WHERE (title ILIKE %s OR content ILIKE %s)
            AND (title ILIKE %s OR content ILIKE %s)
            LIMIT 3
        """, (f'%{words[0]}%', f'%{words[0]}%', f'%{words[1]}%', f'%{words[1]}%'))
    
    rows = cur.fetchall()
    status = '✅' if rows else '❌'
    print(f'\n{status} {label[:50]}')
    print(f'   Search: "{search_term}"')
    for r in rows:
        print(f'   → {r[0]} | {r[1]} | {r[2]}')

cur.close()
conn.close()


✅ Turfschip van Breda in 1590
   Search: "Turfschip Breda"
   → S00807 | Legpenning turfschipper Adriaan van Bergen van Leur | Gedenkpenning
   → S05274 | Draaginsigne opening Moerdijkbrug 1936 | Speld [insigne]
   → SMB000013 | List met het Turfschip | Gravure

✅ Het overlijden van Prins Maurits
   Search: "Maurits gestorven"
   → ST03614 | Het overlijden van Prins Maurits | Gravure

✅ Register van het meselaarsgilde
   Search: "Furie van de Houtepen"
   → ST01304 | Alexander Farnese, hertog van Parma | Prent
   → SMB000069 | Portret van Alexander Farnese, hertog van Parma | Gravure
   → SMB000499 | Het Antidotale Jagt | Prent

✅ portret van lodewijk van Nassau
   Search: "Willem Lodewijk Nassau"
   → ST05084 | Willem Lodewijk, Prins van Nassau | Prent
   → 0008164 | Wendingen jaargang 6, nummer 8, 1924, W. M. Dudok | Tijdschrift
   → ST04703 | Koning Willem II | Prent


In [22]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

cur.execute("""
    SELECT identifier, title, object_type, description, content
    FROM knowledge_base 
    WHERE creator ILIKE '%Swijs%'
    OR title ILIKE '%24-pond%'
    OR title ILIKE '%modelkanon%'
    OR description ILIKE '%Swijs%'
    OR description ILIKE '%24-pond%'
    OR description ILIKE '%wapen van Breda%kanon%'
    LIMIT 5
""")
for r in cur.fetchall():
    print(f'{r[0]} | {r[1]} | {r[2]}')
    print(f'  {r[3][:200] if r[3] else "NULL"}')
    print()

cur.close()
conn.close()

S00276 | Klein kanon, saluutkanon | Kanon
  Klein ceremonieelkanon, op affuit versierd met ornamenten op randen en het wapen van Breda. Afkomstig van het jacht van de Magistraat in Breda. Het staat model voor de 24 ponds kanonnen, waarmee op de



In [23]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

cur.execute("""
    SELECT identifier, title, object_type, description, content, 
           creator, date_created, collection, medium, extent
    FROM knowledge_base 
    WHERE identifier = 'S00276'
""")
r = cur.fetchone()
cols = [d[0] for d in cur.description]
for col, val in zip(cols, r):
    if val and not col.endswith('embedding'):
        print(f'{col:<20} {str(val)[:300]}')

cur.close()
conn.close()

identifier           S00276
title                Klein kanon, saluutkanon
object_type          Kanon
description          Klein ceremonieelkanon, op affuit versierd met ornamenten op randen en het wapen van Breda. Afkomstig van het jacht van de Magistraat in Breda. Het staat model voor de 24 ponds kanonnen, waarmee op de vestingwerken van de stad geschoten werd. In het Stadhuis van Breda staan nog twee van deze kanonne
content              Klein kanon, saluutkanon Johan Swijs Klein ceremonieelkanon, op affuit versierd met ornamenten op randen en het wapen van Breda. Afkomstig van het jacht van de Magistraat in Breda. Het staat model voor de 24 ponds kanonnen, waarmee op de vestingwerken van de stad geschoten werd. In het Stadhuis van 
creator              Johan Swijs
date_created         1697
collection           Stedelijk Museum Breda
medium               Metaal, Hout
extent               breedte: 50 cm, diepte: 96 cm, hoogte: 40 cm


In [13]:
import requests, json, base64, os
import cv2
from pathlib import Path

BACKEND = 'http://194.171.191.226:3414'

LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Nassau',
    'Maurice of Oranje-Nassau': 'Maurits van Nassau',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem',
    'Money of necessity minted during the siege of breda': 'Noodmunt Breda',
    'Noodmunten': 'Noodmunt Breda',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'Model of a 24-pound gun': 'kanon Breda',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht Spaanse bezetting',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht Spaanse garnizoen',
    'Military Operations in flanders with siege and capture of breda': 'beleg van Breda Spinola',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker beleg Breda',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'Turfschip van Breda',
    'Turfschip van Breda in 1590': 'Turfschip van Breda',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'overgave van Breda',
    'collectie 17de en 18de eeuws metaal militair tuig': 'militair tuig',
    'Sporen van de Nassaus': 'Sporen Nassaus',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Jan van Nassau',
    'portret van lodewijk van Nassau': 'Lodewijk van Nassau',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Het overlijden van Prins Maurits': 'overlijden Prins Maurits',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik van Nassau',
}

def classify_image(img_bytes):
    resp = requests.post(
        f'{BACKEND}/api/classify',
        files={'image': ('frame.jpg', img_bytes, 'image/jpeg')},
        timeout=30
    )
    data = resp.json()
    return {
        'label': data.get('landmark', 'unknown'),
        'confidence': data.get('confidence', 0),
        'status': data.get('status', 'unknown')
    }

def get_rag_response(label):
    parts = []
    with requests.post(
        f'{BACKEND}/api/chat',
        json={'message': label, 'landmark': label, 'conversation_id': 'notebook-test', 'audience': 'tourist'},
        stream=True, timeout=60
    ) as resp:
        for line in resp.iter_lines():
            if line:
                try:
                    d = json.loads(line.decode('utf-8') if isinstance(line, bytes) else line)
                    if d.get('type') == 'answer':
                        parts.append(d.get('content', ''))
                except:
                    pass
    return ''.join(parts).strip()

def test_artifact(source, expected_label=None):
    """Test with URL, local image path, or local video path."""

    if source.startswith('http'):
        img_bytes = requests.get(source, timeout=10).content
        r = classify_image(img_bytes)
        best_label = r['label']
        best_conf = r['confidence']
        best_status = r['status']

    elif source.lower().endswith(('.mp4', '.avi', '.mov')):
        cap = cv2.VideoCapture(source)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        max_secs = min(10, int(total_frames / fps))

        best_label, best_conf, best_status = 'unknown', 0, ''

        for sec in range(max_secs):
            pos = int(fps * sec)
            cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
            ret, frame = cap.read()
            if not ret:
                continue
            _, buf = cv2.imencode('.jpg', frame)
            r = classify_image(buf.tobytes())
            lbl = r['label']
            conf = r['confidence']
            status = r['status']
            if lbl != 'unknown' and conf > best_conf:
                best_conf = conf
                best_label = lbl
                best_status = status

        cap.release()

    else:
        img_bytes = Path(source).read_bytes()
        r = classify_image(img_bytes)
        best_label = r['label']
        best_conf = r['confidence']
        best_status = r['status']

    correct = (best_label == expected_label) if expected_label else None
    symbol = '✅' if correct else ('❌' if correct is False else '➡️')

    print(f'{symbol} Label: {best_label} ({best_conf:.3f}) [{best_status}]')
    if expected_label and not correct:
        print(f'   Expected: {expected_label}')
    print(f'   DB term: {LABEL_TO_DB.get(best_label, best_label)}')

    if best_label != 'unknown' and best_conf >= 0.3:
        response = get_rag_response(best_label)
        print(f'   RAG: {response[:150]}...' if len(response) > 150 else f'   RAG: {response}')

    return best_label, best_conf, correct

# ── Batch test all artifact folders ───────────────────────────
base_folder = r'C:\Users\jimal\OneDrive\Documents\UGUIDU\Database Operations\DatasetsAlexi'

results = []
folders = sorted(os.listdir(base_folder))

for artifact_folder in folders:
    folder_path = os.path.join(base_folder, artifact_folder)
    if not os.path.isdir(folder_path):
        continue

    files = os.listdir(folder_path)
    videos = [f for f in files if f.lower().endswith(('.mp4', '.avi', '.mov'))]
    images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    test_file = videos[0] if videos else (images[0] if images else None)
    if not test_file:
        print(f'⚠️  {artifact_folder} — no files')
        continue

    print(f'\n{"="*60}')
    print(f'ARTIFACT: {artifact_folder}')
    source = os.path.join(folder_path, test_file)
    label, conf, correct = test_artifact(source, expected_label=artifact_folder)
    results.append({
        'artifact': artifact_folder,
        'predicted': label,
        'confidence': conf,
        'correct': correct
    })

# ── Summary ───────────────────────────────────────────────────
print(f'\n{"="*60}')
print('BATCH TEST SUMMARY')
print(f'{"="*60}')
correct_count = sum(1 for r in results if r['correct'])
total = len(results)
print(f'Correct: {correct_count}/{total} ({correct_count/total*100:.1f}%)')
print(f'\nFailed:')
for r in results:
    if not r['correct']:
        print(f'  ❌ {r["artifact"][:40]:<40} → {r["predicted"]} ({r["confidence"]:.3f})')


ARTIFACT: Albrecht van Oostenrijk
✅ Label: Albrecht van Oostenrijk (0.888) [SUCCESS]
   DB term: Albrecht van Oostenrijk
   RAG: Albrecht van Oostenrijk werd in 1596 gouverneur van de Nederlanden. Hij trouwde met Isabella, dochter van Filips II. Hij bereikte een wapenstilstand t...

ARTIFACT: Ambrogio Spinola
✅ Label: Ambrogio Spinola (0.885) [SUCCESS]
   DB term: Ambrogio Spinola
   RAG: Ambrogio Spinola was a top Spanish general who recaptured Breda in 1625. He avoided a violent siege by surrounding the city. This led to the surrender...

ARTIFACT: Beeld vna de turfschipper
❌ Label: Equestrian statue of William of Orange (0.854) [SUCCESS]
   Expected: Beeld vna de turfschipper
   DB term: ruiterstandbeeld Willem
   RAG: Breda has equestrian statues of William of Orange. One bronze statue by Émilien de Nieuwerkerke shows William in 1560. Another by A. Schreurs depicts ...

ARTIFACT: City map of Breda in relief
✅ Label: City map of Breda in relief (0.830) [UNCERTAIN]
   DB term: Stads

In [59]:
import requests, json, base64, os
import cv2
from pathlib import Path

BACKEND = 'http://194.171.191.226:3414'

LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Nassau',
    'Maurice of Oranje-Nassau': 'Maurits van Nassau',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem',
    'Money of necessity minted during the siege of breda': 'Noodmunt Breda',
    'Noodmunten': 'Noodmunt Breda',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'Model of a 24-pound gun': 'kanon Breda',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht Spaanse bezetting',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht Spaanse garnizoen',
    'Military Operations in flanders with siege and capture of breda': 'beleg van Breda Spinola',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker beleg Breda',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'Turfschip van Breda',
    'Turfschip van Breda in 1590': 'Turfschip van Breda',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'overgave van Breda',
    'collectie 17de en 18de eeuws metaal militair tuig': 'militair tuig',
    'Sporen van de Nassaus': 'Sporen Nassaus',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Jan van Nassau',
    'portret van lodewijk van Nassau': 'Lodewijk van Nassau',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Het overlijden van Prins Maurits': 'overlijden Prins Maurits',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik van Nassau',
}

def classify_image(img_bytes):
    resp = requests.post(
        f'{BACKEND}/api/classify',
        files={'image': ('frame.jpg', img_bytes, 'image/jpeg')},
        timeout=30
    )
    data = resp.json()
    return {
        'label': data.get('landmark', 'unknown'),
        'confidence': data.get('confidence', 0),
        'status': data.get('status', 'unknown')
    }

def get_rag_response(label):
    parts = []
    with requests.post(
        f'{BACKEND}/api/chat',
        json={'message': label, 'landmark': label, 'conversation_id': 'notebook-test', 'audience': 'tourist'},
        stream=True, timeout=60
    ) as resp:
        for line in resp.iter_lines():
            if line:
                try:
                    d = json.loads(line.decode('utf-8') if isinstance(line, bytes) else line)
                    if d.get('type') == 'answer':
                        parts.append(d.get('content', ''))
                except:
                    pass
    return ''.join(parts).strip()

def test_artifact(source, expected_label=None):
    """Test with URL, local image path, or local video path — tries all videos in folder."""

    if source.startswith('http'):
        img_bytes = requests.get(source, timeout=10).content
        r = classify_image(img_bytes)
        best_label = r['label']
        best_conf = r['confidence']
        best_status = r['status']

    elif source.lower().endswith(('.mp4', '.avi', '.mov')):
        cap = cv2.VideoCapture(source)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_secs = int(total_frames / fps) if fps > 0 else 0
        max_secs = min(30, duration_secs)  # up to 30 seconds

        best_label, best_conf, best_status = 'unknown', 0, ''

        for sec in range(max_secs):
            pos = int(fps * sec)
            cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
            ret, frame = cap.read()
            if not ret:
                continue
            _, buf = cv2.imencode('.jpg', frame)
            r = classify_image(buf.tobytes())
            lbl = r['label']
            conf = r['confidence']
            status = r['status']
            if lbl != 'unknown' and conf > best_conf:
                best_conf = conf
                best_label = lbl
                best_status = status
            # Early exit if we get a very high confidence result
            if conf >= 0.92 and lbl != 'unknown':
                print(f'   Early exit at {sec}s — confidence {conf:.3f}')
                break

        cap.release()

    else:
        img_bytes = Path(source).read_bytes()
        r = classify_image(img_bytes)
        best_label = r['label']
        best_conf = r['confidence']
        best_status = r['status']

    correct = (best_label == expected_label) if expected_label else None
    symbol = 'CORRECT' if correct else ('WRONG' if correct is False else '?')

    print(f'[{symbol}] Label: {best_label} ({best_conf:.3f}) [{best_status}]')
    if expected_label and not correct:
        print(f'   Expected: {expected_label}')
    print(f'   DB term: {LABEL_TO_DB.get(best_label, best_label)}')

    if best_label not in ('unknown', None) and best_conf >= 0.3:
        response = get_rag_response(best_label)
        print(f'   RAG: {response[:150]}...' if len(response) > 150 else f'   RAG: {response}')

    return best_label, best_conf, correct

# Batch test all artifact folders — try up to 3 videos per folder
base_folder = r'C:\Users\jimal\OneDrive\Documents\UGUIDU\Database Operations\DatasetsAlexi'

results = []
folders = sorted(os.listdir(base_folder))

for artifact_folder in folders:
    folder_path = os.path.join(base_folder, artifact_folder)
    if not os.path.isdir(folder_path):
        continue

    files = os.listdir(folder_path)
    videos = sorted([f for f in files if f.lower().endswith(('.mp4', '.avi', '.mov'))])
    images = sorted([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    test_files = videos[:3] if videos else images[:3]
    if not test_files:
        print(f'SKIP: {artifact_folder} — no files')
        continue

    print(f'\n{"="*60}')
    print(f'ARTIFACT: {artifact_folder} ({len(test_files)} file(s) tested)')

    best_label_all, best_conf_all, best_correct_all, best_status_all = 'unknown', 0, False, ''

    for test_file in test_files:
        source = os.path.join(folder_path, test_file)
        print(f'  File: {test_file}')
        label, conf, correct = test_artifact(source, expected_label=artifact_folder)
        if conf > best_conf_all:
            best_conf_all = conf
            best_label_all = label
            best_correct_all = correct
            best_status_all = 'SUCCESS' if conf >= 0.85 else ('UNCERTAIN' if conf >= 0.60 else 'UNKNOWN')

    results.append({
        'artifact': artifact_folder,
        'predicted': best_label_all,
        'confidence': best_conf_all,
        'correct': best_correct_all,
        'status': best_status_all
    })

# Summary
print(f'\n{"="*60}')
print('BATCH TEST SUMMARY')
print(f'{"="*60}')
correct_count = sum(1 for r in results if r['correct'])
total = len(results)
print(f'Correct: {correct_count}/{total} ({correct_count/total*100:.1f}%)')

print('\nPRESENTATION READY — CORRECT AND >85% CONFIDENCE')
for r in results:
    if r['confidence'] >= 0.85 and r['correct']:
        print(f'  [OK] {r["artifact"][:50]:<50} ({r["confidence"]:.3f})')

print('\nUSABLE — CORRECT AND 70-85% CONFIDENCE')
for r in results:
    if 0.70 <= r['confidence'] < 0.85 and r['correct']:
        print(f'  [OK] {r["artifact"][:50]:<50} ({r["confidence"]:.3f})')

print('\nWRONG CLASSIFICATION — >70% CONFIDENCE BUT WRONG LABEL')
for r in results:
    if r['correct'] is False and r['confidence'] >= 0.70:
        predicted = r['predicted'] or 'None'
        print(f'  [WRONG] {r["artifact"][:40]:<40} -> {predicted[:30]} ({r["confidence"]:.3f})')

print('\nRISKY — BELOW 70% OR UNKNOWN')
for r in results:
    if r['confidence'] < 0.70:
        predicted = r['predicted'] or 'None'
        print(f'  [RISKY] {r["artifact"][:45]:<45} ({r["confidence"]:.3f}) -> {predicted[:30]}')


ARTIFACT: Albrecht van Oostenrijk (1 file(s) tested)
  File: 20251212_123531.mp4
[CORRECT] Label: Albrecht van Oostenrijk (0.888) [SUCCESS]
   DB term: Albrecht van Oostenrijk
   RAG: Helaas heb ik geen informatie over Albrecht van Oostenrijk in mijn kennisbank. Wil je liever praten over Frederik Hendrik of andere Bredase historisch...

ARTIFACT: Ambrogio Spinola (1 file(s) tested)
  File: 20251212_123442.mp4
[CORRECT] Label: Ambrogio Spinola (0.885) [SUCCESS]
   DB term: Ambrogio Spinola
   RAG: Helaas heb ik geen informatie over Ambrogio Spinola in mijn kennisbank. Wil je liever praten over Frederik Hendrik of andere Bredase historische figur...

ARTIFACT: Beeld vna de turfschipper (1 file(s) tested)
  File: 20251212_122735.mp4
[WRONG] Label: Equestrian statue of William of Orange (0.854) [SUCCESS]
   Expected: Beeld vna de turfschipper
   DB term: ruiterstandbeeld Willem
   RAG: What do you notice about this equestrian statue of William of Orange?

ARTIFACT: City map of Breda in re

In [60]:
import cv2, requests

# Use a known good video
cap = cv2.VideoCapture(r'C:\Users\jimal\OneDrive\Documents\UGUIDU\Database Operations\DatasetsAlexi\Ambrogio Spinola\20251212_123442.mp4')
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
ret, frame = cap.read()
cap.release()

_, buf = cv2.imencode('.jpg', frame)
img_bytes = buf.tobytes()

resp = requests.post(
    'http://194.171.191.226:3414/api/classify',
    files={'image': ('frame.jpg', img_bytes, 'image/jpeg')},
    timeout=30
)

import json
print(json.dumps(resp.json(), indent=2))

{
  "action": "ask_user_confirm",
  "confidence": 0.8341042876243592,
  "inference_time_ms": 302,
  "landmark": "Ambrogio Spinola",
  "message": "Is this Ambrogio Spinola?",
  "status": "UNCERTAIN"
}


In [61]:
import cv2, requests, json

cap = cv2.VideoCapture(r'C:\Users\jimal\OneDrive\Documents\UGUIDU\Database Operations\DatasetsAlexi\Justinus of Nassau\20251212_122702.mp4')
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
ret, frame = cap.read()
cap.release()

_, buf = cv2.imencode('.jpg', frame)
resp = requests.post(
    'http://194.171.191.226:3414/api/classify',
    files={'image': ('frame.jpg', buf.tobytes(), 'image/jpeg')},
    timeout=30
)
print(json.dumps(resp.json(), indent=2))

{
  "action": "ask_user_confirm",
  "confidence": 0.7985333323478698,
  "inference_time_ms": 10318,
  "landmark": "Justinus of Nassau",
  "message": "Is this Justinus of Nassau?",
  "status": "UNCERTAIN",
  "top_3": [
    {
      "confidence": 0.7985,
      "label": "Justinus of Nassau"
    },
    {
      "confidence": 0.7806,
      "label": "Frederick Henry of Orange-Nassau"
    },
    {
      "confidence": 0.7544,
      "label": "Maurice of Oranje-Nassau"
    }
  ]
}


## DATABASE EVALUTION

In [14]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

search_terms = {
    'Schelpdieren en vis': 'Schelpdieren',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht garnizoen',
    'Register van het meselaarsgilde': 'meselaarsgilde',
    'Radslotpistool van Charles de Heraughières': 'Radslotpistool',
    'portret van lodewijk van Nassau': 'Lodewijk Nassau',
    'Portret van Jan de Jongere': 'Jan de Jongere',
    'partizaan': 'Partizaan',
    'ONZEKER': 'onbekend',
    'Obsidio Bredana': 'Obsidio Bredana',
    'Noodmunten': 'Noodmunt',
    'Money of necessity minted during the siege of breda': 'Noodmunt',
    'Model van het Turfschip': 'Turfschip',
    'Model of a 24-pound gun': 'kanon',
    'Military Operations in flanders': 'beleg Breda',
    'Maurice of Oranje': 'Maurits',
    'Justinus of Nassau': 'Justinus van Nassau',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Het overlijden van Prins Maurits': 'overlijden Maurits',
    'het beleg van Breda': 'beleg van Breda',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik',
    'Frederick Henry of Orange': 'Frederik Hendrik',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Departure of the Spanish occupation': 'Uittocht Spaanse',
    'De overgave van breda': 'overgave Breda',
    'Constantijn Huygens': 'Constantijn Huygens',
    'collectie 17de en 18de eeuws metaal militair tuig': 'militair tuig',
    'City map of Breda in relief': 'Stadsplan relief',
    'Beeld van de turfschipper': 'turfschipper',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
}

results = []
for artifact, term in search_terms.items():
    cur.execute("""
        SELECT identifier, title, object_type, 
               LEFT(description, 200) as description,
               LEFT(content, 300) as content,
               date_created, collection
        FROM knowledge_base 
        WHERE title ILIKE %s 
        ORDER BY 
            CASE WHEN identifier LIKE 'BR%%' THEN 0 ELSE 1 END,
            identifier
        LIMIT 5
    """, (f'%{term}%',))
    rows = cur.fetchall()
    
    if rows:
        for r in rows:
            results.append({
                'artifact': artifact,
                'search_term': term,
                'identifier': r[0],
                'title': r[1],
                'object_type': r[2],
                'description': r[3],
                'content': r[4],
                'date_created': r[5],
                'collection': r[6],
                'found': True
            })
    else:
        results.append({
            'artifact': artifact,
            'search_term': term,
            'identifier': 'NONE',
            'title': 'NO MATCH',
            'object_type': None,
            'description': None,
            'content': None,
            'date_created': None,
            'collection': None,
            'found': False
        })

cur.close()
conn.close()

df = pd.DataFrame(results)

# Summary
print('=== DB COVERAGE SUMMARY ===')
summary = df.groupby('artifact').agg(
    found=('found', 'first'),
    record_count=('identifier', 'count'),
    identifiers=('identifier', lambda x: ', '.join(x.tolist()[:3]))
).reset_index()

for _, row in summary.iterrows():
    status = '✅' if row['found'] else '❌'
    print(f'{status} {row["artifact"][:50]:<50} → {row["record_count"]} records | {row["identifiers"]}')

# Full details
print('\n\n=== FULL DB RECORDS ===')
for artifact in search_terms.keys():
    subset = df[df['artifact'] == artifact]
    print(f'\n{"="*60}')
    print(f'ARTIFACT: {artifact}')
    for _, row in subset.iterrows():
        if row['found']:
            print(f'  {row["identifier"]} | {row["title"]} | {row["object_type"]}')
            print(f'  DESC: {row["description"]}')
            print(f'  CONTENT: {row["content"]}')
        else:
            print(f'  ❌ NO DB RECORDS FOUND')

=== DB COVERAGE SUMMARY ===
✅ Albrecht van Oostenrijk                            → 1 records | ST02224
✅ Ambrogio Spinola                                   → 3 records | ST00588, ST00629, ST02230
✅ Beeld van de turfschipper                          → 4 records | S00274, S00525, S00807
❌ City map of Breda in relief                        → 1 records | NONE
✅ Constantijn Huygens                                → 1 records | 0003009
❌ De overgave van breda                              → 1 records | NONE
❌ Departure of the Spanish occupation                → 1 records | NONE
✅ Filips Willem, prins van Oranje                    → 5 records | BR00032, S00707, ST03462
✅ Frederick Henry of Orange                          → 5 records | BR00013, GT00014, GT01748
✅ Frederik Hendrik, Prins van Oranje                 → 5 records | BR00013, GT00014, GT01748
✅ Herdenkingsbeker van het beleg van Breda in 1637   → 1 records | BR00048
❌ Het overlijden van Prins Maurits                   → 1 records | NON

In [15]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

missing = {
    'Retreat of the Spanish Garrison from Breda': ['Uittocht', 'Spaans garnizoen', 'Spaanse bezetting', 'terugtocht', 'garnizoen Breda'],
    'Register van het meselaarsgilde': ['meselaarsgilde', 'Furie van Houtepen', 'meselaars', 'gilde Breda', 'burgers omgekomen'],
    'portret van lodewijk van Nassau': ['Lodewijk van Nassau', 'Lodewijk Nassau', 'Louis van Nassau', 'portret Lodewijk'],
    'Portret van Jan de Jongere': ['Jan de Jongere', 'graaf Nassau Siegen', 'Jan Nassau Siegen', 'Nassau Siegen'],
    'Military Operations in flanders': ['Militaire operaties', 'Operations in Flanders', 'Vlaanderen beleg', 'militaire Breda', 'siege capture'],
    'Het overlijden van Prins Maurits': ['overlijden Maurits', 'dood Maurits', 'Maurits gestorven', 'prins Maurits 1625'],
    'Departure of the Spanish occupation': ['Uittocht Spaanse', 'vertrek Spanjaarden', 'Spaanse bezetting vertrek', 'oktober 1637'],
    'De overgave van breda': ['overgave Breda', 'Las Lanzas', 'surrender Breda', 'Velazquez', 'capitulatie'],
    'collectie 17de en 18de eeuws metaal militair tuig': ['militair tuig', 'wapentuig', 'metalen militair', '17e eeuw wapen', 'collectie wapens'],
    'City map of Breda in relief': ['Stadsplan relief', 'reliëf Breda', 'plattegrond relief', 'stadsmodel', 'Breda model stad'],
}

for artifact, terms in missing.items():
    print(f'\n{"="*60}')
    print(f'SEARCHING: {artifact}')
    found_any = False
    for term in terms:
        cur.execute(
            "SELECT identifier, title, object_type FROM knowledge_base WHERE title ILIKE %s OR description ILIKE %s LIMIT 3",
            (f'%{term}%', f'%{term}%')
        )
        rows = cur.fetchall()
        if rows:
            found_any = True
            for r in rows:
                print(f'  ✅ "{term}" → {r[0]} | {r[1]} | {r[2]}')
    if not found_any:
        print(f'  ❌ Nothing found for any search term')

cur.close()
conn.close()


SEARCHING: Retreat of the Spanish Garrison from Breda
  ✅ "Uittocht" → ST06931 | Uittocht van de Spaanse bezetting | Tekening
  ✅ "Uittocht" → ST06967 | Inname van Breda door Maurits, 4 maart 1590 | Prent
  ✅ "Uittocht" → GT01748 | Kaart van het beleg van Breda door Frederik Hendrik | Prent
  ✅ "Spaanse bezetting" → GT01748 | Kaart van het beleg van Breda door Frederik Hendrik | Prent
  ✅ "Spaanse bezetting" → ST01626 | Kaart van het beleg van Breda door Frederik Hendrik in 1637 | Prent
  ✅ "Spaanse bezetting" → S01073 | De uittocht der Spaanse bezetting uit Breda op 10 oktober 1637 | Schilderij

SEARCHING: Register van het meselaarsgilde
  ✅ "Furie van Houtepen" → SMB000069 | Portret van Alexander Farnese, hertog van Parma | Gravure
  ✅ "gilde Breda" → S02181 | Draagpenning Brandewijnstokersgilde Breda | Onderscheiding
  ✅ "gilde Breda" → S02182 | Draagpenning Brandewijnstokersgilde Breda | Onderscheiding
  ✅ "gilde Breda" → S02186 | Draagpenning Brandewijnstokersgilde Breda | Onders

In [17]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

cur.execute("""
    SELECT identifier, title, object_type, description
    FROM knowledge_base 
    WHERE creator ILIKE '%Verhoeven%'
    OR creator ILIKE '%Ravesteyn%'
    OR creator ILIKE '%Cornelis van Bergen%'
    OR title ILIKE '%Furie%'
    OR title ILIKE '%Haultepenne%'
    OR title ILIKE '%Jan de Jongere%'
    OR title ILIKE '%Jan van Nassau-Siegen%'
    OR description ILIKE '%Francisco Pieri%'
    LIMIT 10
""")
for r in cur.fetchall():
    print(f'{r[0]} | {r[1]} | {r[2]}')
    print(f'  {r[3][:150] if r[3] else "NULL"}')
    print()

cur.close()
conn.close()

S06108 | Sierband | Armband
  Ronde ajourband, bevestigd op koperen ondergrond, voorzien van rechthoekig plaatje met gegraveerde tekst.

ST00052 | De Furie van de Houtepen in 1581 | Gravure
  Het is een complete verrassing voor de Bredanaars als Spaanse soldaten 's nachts de stad binnendringen. Het kasteel is als eerste aan de beurt; dat wo

ST00065 | Spotprent op de financiering van het beleg van Breda door Spinola | Prent
  De ruiter op de spotprent is legerleiding Spinola, letterlijk met de handen in het haar. Tussen de benen van zijn paard is de Spaanse koning te zien, 

0005546 | Denk hier ook eens over na... | Affiche
  Affiche voor personeelswerving van PTT: 'Denk hier ook eens over na... Met het einddiploma HBS (A of B) of Gymnasium (α of β) kunt u in aanmerking kom

0006417 | Eerste dag envelop: Amphilex 1967 | Enveloppe
  Envelop bedrukt met de tekst 'Amphilex 67, eerste dag van uitgifte - first day of issue'. Daarbij een logo van Amphilex. Op de envelop zijn drie postz

B014

In [18]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

cur.execute("""
    SELECT identifier, title, object_type
    FROM knowledge_base 
    WHERE title ILIKE '%Ravesteyn%'
    OR title ILIKE '%Jan van Nassau%'
    OR creator ILIKE '%Ravesteyn%'
    OR title ILIKE '%overlijden Maurits%'
    OR creator ILIKE '%Verhoeven%'
    LIMIT 10
""")
for r in cur.fetchall():
    print(f'{r[0]} | {r[1]} | {r[2]}')

cur.close()
conn.close()

ST00430 | Kaart van het beleg van Breda door Spinola | Prent
ST00431 | Kaart van het beleg van Breda door Spinola | Prent
ST00669 | Kaart van het beleg van Breda door Spinola | Prent
ST01604 | Kaart van het beleg van Breda door Spinola | Prent
ST03614 | Het overlijden van Prins Maurits | Gravure
ST04405 | Kaart van het beleg van Breda door Spinola | Prent
B01438 | Twee zilveren ampullen | Ampul
B01441 | Twee zilveren altaarschellen | Altaarschel
BR00003 | Justinus van Nassau | Schilderij
BT03328 | Gekruisigde Christus met gebed over Geloof, Hoop en Liefde. | Gebedsprent


In [19]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# All verified search terms for each artifact
artifact_searches = {
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Beeld van de turfschipper': 'turfschipper Adriaan',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'Constantijn Huygens': 'Constantijn Huygens Illustere',
    'De overgave van breda': 'Las Lanzas',
    'Departure of the Spanish occupation': 'Uittocht Spaanse bezetting',
    'Filips Willem, prins van Oranje': 'Filips Willem prins Oranje',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Oranje Nassau',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik van Oranje Nassau',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker beleg Breda',
    'Het overlijden van Prins Maurits': 'overlijden Prins Maurits',
    'het beleg van Breda': 'beleg van Breda Spinola',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Justinus of Nassau': 'Justinus van Nassau',
    'Maurice of Oranje-Nassau': 'Maurits prins van Oranje-Nassau',
    'Military Operations in flanders': 'beleg van Breda Spinola',
    'Model of a 24-pound gun': 'kanon Breda Héraugière',
    'Model van het Turfschip': 'Turfschip Breda 1590',
    'Money of necessity / Noodmunten': 'Noodmunt Breda stuiver',
    'Obsidio Bredana': 'Obsidio Bredana',
    'partizaan': 'Partizaan 17de eeuw',
    'Portret van Jan de Jongere': 'Jan van Nassau-Siegen',
    'portret van lodewijk van Nassau': 'Willem Lodewijk van Nassau',
    'Radslotpistool van Charles de Heraughières': 'Radslotpistool',
    'Register van het meselaarsgilde': 'Furie Houtepen',
    'Retreat of the Spanish Garrison': 'Uittocht Spaanse bezetting',
    'Schelpdieren en vis': 'Schelpdieren',
    'collectie 17de en 18de eeuws metaal militair tuig': 'halfharnas rapier militair',
    'Turfschip van Breda in 1590': 'Turfschip van Breda',
}

print('='*80)
print('FULL DB COVERAGE REPORT FOR ALL MUSEUM ARTIFACTS')
print('='*80)

all_results = []

for artifact, search_term in artifact_searches.items():
    words = search_term.split()
    
    # Build dynamic query with all words
    conditions = ' AND '.join([f"(title ILIKE %s OR description ILIKE %s OR content ILIKE %s)" for _ in words])
    params = []
    for w in words:
        params.extend([f'%{w}%', f'%{w}%', f'%{w}%'])
    
    query = f"""
        SELECT identifier, title, object_type, 
               LEFT(description, 300) as description,
               LEFT(content, 300) as content,
               date_created, creator
        FROM knowledge_base 
        WHERE {conditions}
        ORDER BY 
            CASE WHEN identifier LIKE 'BR%%' THEN 0 
                 WHEN identifier LIKE 'S%%' THEN 1
                 ELSE 2 END,
            identifier
        LIMIT 3
    """
    
    cur.execute(query, params)
    rows = cur.fetchall()
    
    status = '✅' if rows else '❌'
    print(f'\n{status} {artifact}')
    print(f'   Search: "{search_term}"')
    
    if rows:
        for r in rows:
            print(f'   {r[0]} | {r[1]} | {r[2]} | {r[5]}')
            print(f'   Creator: {r[6]}')
            print(f'   Desc: {r[3][:200] if r[3] else "NULL"}')
            all_results.append({
                'artifact': artifact,
                'search_term': search_term,
                'found': True,
                'identifier': r[0],
                'title': r[1],
                'object_type': r[2],
                'description': r[3],
                'content': r[4],
                'date_created': r[5],
                'creator': r[6]
            })
    else:
        print(f'   ❌ NO RECORDS FOUND IN DB')
        all_results.append({
            'artifact': artifact,
            'search_term': search_term,
            'found': False,
            'identifier': None,
            'title': None,
            'object_type': None,
            'description': None,
            'content': None,
            'date_created': None,
            'creator': None
        })

cur.close()
conn.close()

# Summary
df = pd.DataFrame(all_results)
found = df[df['found']==True]['artifact'].nunique()
total = df['artifact'].nunique()

print('\n' + '='*80)
print(f'SUMMARY: {found}/{total} artifacts have DB records')
print('='*80)
print('\nMISSING:')
for a in df[df['found']==False]['artifact'].unique():
    print(f'  ❌ {a}')

print('\nFOUND:')
for a in df[df['found']==True]['artifact'].unique():
    ids = df[(df['artifact']==a) & (df['found']==True)]['identifier'].tolist()
    print(f'  ✅ {a} → {", ".join(ids)}')

FULL DB COVERAGE REPORT FOR ALL MUSEUM ARTIFACTS

✅ Albrecht van Oostenrijk
   Search: "Albrecht van Oostenrijk"
   SMB000419 | Isabella Clara Eugenia van Spanje | Schilderij | 1629 – 1750
   Creator: Peter Paul Rubens, anoniem, Anthony van Dyck
   Desc: Isabella Clara Eugenia is een dochter van de Spaanse koning Filips II. Zij en haar man, aartshertog Albrecht van Oostenrijk, worden in 1598 landvoogdes en landvoogd van de Zuidelijke Nederlanden: Fili
   ST02224 | Albrecht van Oostenrijk | Prent | 1644
   Creator: Jonas Suyderhoef
   Desc: Filips II, koning van Spanje, benoemt zijn neef Albrecht van Oostenrijk in 1596 tot gouverneur van de Nederlanden. Albrecht, dan nog onderkoning van Portugal en bisschop van Toledo, trouwt met Filips’
   G13104 | Devotiestolp met de eik van Scherpenheuvel en Maria | Glazen stolp | 1850 – 1900
   Creator: None
   Desc: Devotiestolp met de eik van Scherpenheuvel met Mariabeeld met aan weerszijden geknield Aartshertog Albrecht van Oostenrijk en Isabella

## LABELS TESTING

In [30]:
# Final verified single-word search terms
FINAL_LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Oranje Nassau',
    'Maurice of Oranje-Nassau': 'Maurits prins van Oranje-Nassau',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem',
    'Money of necessity minted during the siege of breda': 'Noodmunt',
    'Noodmunten': 'Noodmunt',
    'City map of Breda in relief': 'Stadsplan',
    'Model of a 24-pound gun': 'saluutkanon',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper Adriaan',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht Spaanse',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht Spaanse',
    'Military Operations in flanders with siege and capture of breda': 'Obsidio Bredana',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'List met het Turfschip',
    'Turfschip van Breda in 1590': 'List met het Turfschip',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'Las Lanzas',
    'collectie 17de en 18de eeuws metaal militair tuig': 'halfharnas',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Nassau-Siegen',
    'portret van lodewijk van Nassau': 'Willem Lodewijk',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Het overlijden van Prins Maurits': 'Maurits gestorven',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen': 'Furie',
    'Sporen van de Nassaus': 'Sporen Nassau',
}

In [31]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

for label, term in FINAL_LABEL_TO_DB.items():
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
        (f'%{term}%', f'%{term}%')
    )
    r = cur.fetchone()
    status = '✅' if r else '❌'
    print(f'{status} {label[:45]:<45} → {term:<30} → {r[1][:40] if r else "NO MATCH"}')

cur.close()
conn.close()

✅ Justinus of Nassau                            → Justinus van Nassau            → Justinus van Nassau
✅ Frederick Henry of Orange-Nassau              → Frederik Hendrik van Oranje Nassau → Portret van Frederik Hendrik van Oranje 
❌ Maurice of Oranje-Nassau                      → Maurits prins van Oranje-Nassau → NO MATCH
❌ Equestrian statue of William of Orange        → ruiterstandbeeld Willem        → NO MATCH
✅ Money of necessity minted during the siege of → Noodmunt                       → Noodmunt Breda 40 stuiver uit 1577
✅ Noodmunten                                    → Noodmunt                       → Noodmunt Breda 40 stuiver uit 1577
✅ City map of Breda in relief                   → Stadsplan                      → Stadsplan in reliëf van de stad Breda
✅ Model of a 24-pound gun                       → saluutkanon                    → Klein kanon, saluutkanon
✅ Constantijn Huygens, curator van de Illustere → Constantijn Huygens            → Postzegelvel, Nederland 75 cent, De 

In [32]:
# Fix the 5 failing ones
fixes = {
    'Maurice of Oranje-Nassau': 'Maurits',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht',
    'Het overlijden van Prins Maurits': 'overlijden Maurits',
    'Sporen van de Nassaus': 'Nassaus Breda',
}

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

for label, term in fixes.items():
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
        (f'%{term}%', f'%{term}%')
    )
    r = cur.fetchone()
    status = '✅' if r else '❌'
    print(f'{status} {label[:45]:<45} → "{term}" → {r[1][:50] if r else "NO MATCH"}')

cur.close()
conn.close()

✅ Maurice of Oranje-Nassau                      → "Maurits" → Mauritssingel en Oranjesingel
✅ Equestrian statue of William of Orange        → "ruiterstandbeeld" → Kasteelplein
✅ Departure of the Spanish occupation from Bred → "Uittocht" → Huwelijkspenning Vermeeren-Limpens
✅ Retreat of the Spanish Garrison from Breda    → "Uittocht" → Huwelijkspenning Vermeeren-Limpens
❌ Het overlijden van Prins Maurits              → "overlijden Maurits" → NO MATCH
❌ Sporen van de Nassaus                         → "Nassaus Breda" → NO MATCH


In [33]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Het overlijden van Prins Maurits - we know ST03614 exists
for term in ['Verhoeven', 'overlijden', 'Maurits 1625', 'nieuwsbericht Maurits']:
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
        (f'%{term}%', f'%{term}%')
    )
    r = cur.fetchone()
    print(f'"{term}" → {r[0] + " | " + r[1] if r else "NO MATCH"}')

print()

# Sporen van de Nassaus
for term in ['Nassaus', 'Sporen van', 'Nassau erfenis', 'Nassau Breda']:
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
        (f'%{term}%', f'%{term}%')
    )
    r = cur.fetchone()
    print(f'"{term}" → {r[0] + " | " + r[1] if r else "NO MATCH"}')

cur.close()
conn.close()

"Verhoeven" → G13494 | Altaarreliek met schedel van soldaat van Thebaanse legioen
"overlijden" → G13416 | Reliekkruis met relieken van het Heilig Kruis en de heilige Armandus van Bordeaux
"Maurits 1625" → NO MATCH
"nieuwsbericht Maurits" → NO MATCH

"Nassaus" → GT02312 | Hoofdingang gevangenis te Breda
"Sporen van" → 0004137 | Rosbeek 57, het huis van San Claessens, in de voetsporen van De Stijl en Het Nieuwe Bouwen.
"Nassau erfenis" → NO MATCH
"Nassau Breda" → NO MATCH


In [34]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Check ST03614 directly
cur.execute("SELECT identifier, title, content FROM knowledge_base WHERE identifier = 'ST03614'")
r = cur.fetchone()
if r:
    print(f'ST03614 found: {r[1]}')
    print(f'Content: {r[2][:300]}')
else:
    print('ST03614 NOT in DB')

# Check Sporen van de Nassaus - what is the actual artifact?
cur.execute("""
    SELECT identifier, title, content 
    FROM knowledge_base 
    WHERE content ILIKE '%Nassau%' AND content ILIKE '%Breda%' AND content ILIKE '%sporen%'
    LIMIT 3
""")
for r in cur.fetchall():
    print(f'\n{r[0]} | {r[1]}')
    print(f'{r[2][:200]}')

cur.close()
conn.close()

ST03614 found: Het overlijden van Prins Maurits
Content: Het overlijden van Prins Maurits Abraham Verhoeven Het overlijden van Prins Maurits door Abraham Verhoeven, de uitgever van dit nieuwsbericht. Hij was een bekende drukker uit de Zuidelijke Nederlanden. Hij berichtte regelmatig over het Beleg van Breda. In 1625 meldt hij ontroerd dat Maurits is gesto


In [35]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Test exact title match
for term in ['overlijden van Prins Maurits', 'overlijden Prins', 'Prins Maurits gestorven', 'Abraham Verhoeven Maurits']:
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
        (f'%{term}%', f'%{term}%')
    )
    r = cur.fetchone()
    print(f'"{term}" → {r[0] + " | " + r[1] if r else "NO MATCH"}')

cur.close()
conn.close()

"overlijden van Prins Maurits" → ST03614 | Het overlijden van Prins Maurits
"overlijden Prins" → NO MATCH
"Prins Maurits gestorven" → NO MATCH
"Abraham Verhoeven Maurits" → NO MATCH


In [36]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

FINAL_LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Oranje Nassau',
    'Maurice of Oranje-Nassau': 'Maurits',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld',
    'Money of necessity minted during the siege of breda': 'Noodmunt',
    'Noodmunten': 'Noodmunt',
    'City map of Breda in relief': 'Stadsplan',
    'Model of a 24-pound gun': 'saluutkanon',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper Adriaan',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht',
    'Military Operations in flanders with siege and capture of breda': 'Obsidio Bredana',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'List met het Turfschip',
    'Turfschip van Breda in 1590': 'List met het Turfschip',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'Las Lanzas',
    'collectie 17de en 18de eeuws metaal militair tuig': 'halfharnas',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Nassau-Siegen',
    'portret van lodewijk van Nassau': 'Willem Lodewijk',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Het overlijden van Prins Maurits': 'overlijden van Prins Maurits',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen': 'Furie',
    'Sporen van de Nassaus': 'Nassaus',
}

print(f'Testing {len(FINAL_LABEL_TO_DB)} labels...\n')
passed = 0
failed = []

for label, term in FINAL_LABEL_TO_DB.items():
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
        (f'%{term}%', f'%{term}%')
    )
    r = cur.fetchone()
    if r:
        passed += 1
        print(f'✅ {label[:50]:<50} → "{term}" → {r[1][:40]}')
    else:
        failed.append((label, term))
        print(f'❌ {label[:50]:<50} → "{term}" → NO MATCH')

cur.close()
conn.close()

print(f'\n{"="*60}')
print(f'PASSED: {passed}/{len(FINAL_LABEL_TO_DB)}')
if failed:
    print(f'FAILED:')
    for l, t in failed:
        print(f'  ❌ {l} → "{t}"')

Testing 33 labels...

✅ Justinus of Nassau                                 → "Justinus van Nassau" → Justinus van Nassau
✅ Frederick Henry of Orange-Nassau                   → "Frederik Hendrik van Oranje Nassau" → Portret van Frederik Hendrik van Oranje 
✅ Maurice of Oranje-Nassau                           → "Maurits" → Maurits, prins van Oranje-Nassau
✅ Equestrian statue of William of Orange             → "ruiterstandbeeld" → Beeld van vogel / staartmees
✅ Money of necessity minted during the siege of bred → "Noodmunt" → Noodmunt Breda 40 stuiver uit 1577
✅ Noodmunten                                         → "Noodmunt" → Noodmunt Breda 40 stuiver uit 1577
✅ City map of Breda in relief                        → "Stadsplan" → Stadsplan in reliëf van de stad Breda
✅ Model of a 24-pound gun                            → "saluutkanon" → Klein kanon, saluutkanon
✅ Constantijn Huygens, curator van de Illustere Scho → "Constantijn Huygens" → Postzegelvel, Nederland 75 cent, De were
✅ Beeld vn

In [37]:
better_terms = {
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Illustere School',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia van Spanje',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen': 'Houtepen 1581',
    'Sporen van de Nassaus': 'Nassau Breda',
}

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

for label, term in better_terms.items():
    cur.execute(
        "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
        (f'%{term}%', f'%{term}%')
    )
    r = cur.fetchone()
    status = '✅' if r else '❌'
    print(f'{status} {label[:45]:<45} → "{term}" → {r[1][:50] if r else "NO MATCH"}')

cur.close()
conn.close()

❌ Equestrian statue of William of Orange        → "ruiterstandbeeld Willem" → NO MATCH
✅ Constantijn Huygens, curator van de Illustere → "Illustere School" → Professor Lodewijk Gerardus van Renesse
✅ Isabella Clara Eugenia van Spanja             → "Isabella Clara Eugenia van Spanje" → Isabella Clara Eugenia van Spanje
❌ Register van het meselaarsgilde met omgekomen → "Houtepen 1581" → NO MATCH
❌ Sporen van de Nassaus                         → "Nassau Breda" → NO MATCH


In [ ]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

tests = {
    'Equestrian statue': ['ruiterstandbeeld', 'Willem III kasteelplein', 'Kasteelplein Willem', 'standbeeld Willem Oranje'],
    'Register meselaarsgilde': ['meselaarsgilde', 'metselaarsgilde', '584 burgers', 'Haultepenne', 'furie 1581'],
    'Sporen van de Nassaus': ['erfenis Nassau', 'Hendrik III Nassau', 'Nassau kasteel', 'nassausche'],
}

for artifact, terms in tests.items():
    print(f'\n{artifact}:')
    for term in terms:
        cur.execute(
            "SELECT identifier, title FROM knowledge_base WHERE title ILIKE %s OR content ILIKE %s LIMIT 1",
            (f'%{term}%', f'%{term}%')
        )
        r = cur.fetchone()
        status = '✅' if r else '❌'
        print(f'  {status} "{term}" → {r[0] + " | " + r[1][:50] if r else "NO MATCH"}')

cur.close()
conn.close()

## Vector database


In [38]:
! pip install sentence-transformers

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 21.9 MB/s  0:00:00
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ---------------------------------------- 10.8/10.8 MB 79.3 MB/s  0:00:00
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl (341 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   -------- ------------------------------ 26.0/123.0 MB 125.6 MB/s eta 0:00:01
   -------------- ------------------------ 45.1/123.0 MB 118.8 MB/s eta 0:00:01
   --------------------- ----------------- 67.9/123.0 MB 106.3 MB/s eta 0:00:01
   ----------------------------- --------- 92.0/123.0 MB 108.1 MB/s eta 0:00:01
   ---------------------------------

In [39]:
import psycopg2
import numpy as np
from sentence_transformers import SentenceTransformer

# Load a small multilingual model
print('Loading embedding model...')
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print('Model loaded!')

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Test queries
test_queries = [
    'Equestrian statue of William of Orange',
    'ruiterstandbeeld Willem van Oranje',
    'Constantijn Huygens Illustere School Breda',
    'Sporen van de Nassaus',
    'Register meselaarsgilde Furie Houtepen',
    'Ambrogio Spinola beleg Breda 1625',
    'Justinus van Nassau gouverneur',
]

for query in test_queries:
    # Embed the query
    query_embedding = model.encode(query).tolist()
    embedding_str = '[' + ','.join(map(str, query_embedding)) + ']'
    
    # Vector similarity search
    cur.execute("""
        SELECT identifier, title, object_type,
               1 - (text_embedding <=> %s::vector) as similarity
        FROM knowledge_base
        WHERE text_embedding IS NOT NULL
        ORDER BY text_embedding <=> %s::vector
        LIMIT 3
    """, (embedding_str, embedding_str))
    
    rows = cur.fetchall()
    print(f'\n🔍 Query: "{query}"')
    for r in rows:
        print(f'   {r[3]:.3f} | {r[0]} | {r[1][:50]} | {r[2]}')

cur.close()
conn.close()

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\jimal\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jimal\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!

🔍 Query: "Equestrian statue of William of Orange"
   0.163 | 0005383 | Antonius en Cleopatra van William Shakespeare door | Affiche
   0.160 | 0000674 | Antonius en Cleopatra | Affiche
   0.156 | 0005994 | Louis Armstrong plays W.C. Handy | Platenhoes

🔍 Query: "ruiterstandbeeld Willem van Oranje"
   0.237 | 0005964 | Silver's blue | Platenhoes
   0.231 | ST03553 | Willem IV op zijn doodsbed | Prent
   0.229 | 0005989 | Jan Corduwener. Dancing time no. 6 | Platenhoes

🔍 Query: "Constantijn Huygens Illustere School Breda"
   0.258 | ST05816 | Seminarie en gymnasium in Lingen | Prent
   0.244 | ST01567 | Portret van Antonius Hulsius | Foto
   0.243 | BT03657 | S. Matheus, lezend in een boek met speer in zijn l | Devotieprent

🔍 Query: "Sporen van de Nassaus"
   0.284 | BR00003 | Justinus van Nassau | Schilderij
   0.276 | B01846 | Driestel: kazuifel, dalmatiek, tuniek met toebehor | Kelkvelum
   0.275 | S01060 | Origineel medaillon van Aristides afkomstig van de | Medaillo

In [40]:
from sentence_transformers import SentenceTransformer
import psycopg2

print('Loading e5-small-v2...')
model = SentenceTransformer('intfloat/e5-small-v2')
print('Loaded!')

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

test_queries = [
    'Equestrian statue of William of Orange',
    'ruiterstandbeeld Willem van Oranje',
    'Ambrogio Spinola siege Breda 1625',
    'Justinus van Nassau governor Breda',
    'Noodmunt emergency coins siege',
    'Constantijn Huygens Illustere School',
    'Sporen van de Nassaus',
]

for query in test_queries:
    # Use correct e5 prefix
    prefixed = f'query: {query}'
    embedding = model.encode(prefixed).tolist()
    emb_str = '[' + ','.join(map(str, embedding)) + ']'
    
    cur.execute("""
        SELECT identifier, title, object_type,
               1 - (text_embedding <=> %s::vector) as similarity
        FROM knowledge_base
        WHERE text_embedding IS NOT NULL
        ORDER BY text_embedding <=> %s::vector
        LIMIT 3
    """, (emb_str, emb_str))
    
    rows = cur.fetchall()
    print(f'\n🔍 "{query}"')
    for r in rows:
        print(f'   {r[3]:.3f} | {r[0]} | {r[1][:50]}')

cur.close()
conn.close()

Loading e5-small-v2...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\jimal\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jimal\.cache\huggingface\hub\models--intfloat--e5-small-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Loaded!

🔍 "Equestrian statue of William of Orange"
   0.819 | SMB000456 | Kaart van het Vorstendom Orange en het Comtat Vena
   0.818 | ST05165 | Loc No. OSM 8, B&R 13 van de Machinefabriek Breda 
   0.817 | ST05164 | Loc No. OSM 8, B&R 13 van de Machinefabriek Breda 

🔍 "ruiterstandbeeld Willem van Oranje"
   0.872 | SMB000416 | Ruiterstandbeeld van Willem van Oranje
   0.866 | G02835 | Theelepel
   0.863 | ST06069 | Ruiterstandbeeld van Willem III van Oranje Nassau 

🔍 "Ambrogio Spinola siege Breda 1625"
   0.894 | ST00431 | Kaart van het beleg van Breda door Spinola
   0.887 | ST00036 | Kaart van het beleg van Breda door Spinola, Siège 
   0.885 | ST00036.2 | Siège de Breda.

🔍 "Justinus van Nassau governor Breda"
   0.851 | BR00003 | Justinus van Nassau
   0.849 | ST01626 | Kaart van het beleg van Breda door Frederik Hendri
   0.848 | ST00792 | Foto van portret Odilia van Nassau

🔍 "Noodmunt emergency coins siege"
   0.822 | S00766 | Noodmunt Breda 40 stuiver 1625
   0.821 | S0338

In [41]:
from sentence_transformers import SentenceTransformer
import psycopg2

print('Loading e5-small-v2...')
model = SentenceTransformer('intfloat/e5-small-v2')
print('Loaded!')

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Oranje Nassau',
    'Maurice of Oranje-Nassau': 'Maurits',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld',
    'Money of necessity minted during the siege of breda': 'Noodmunt',
    'Noodmunten': 'Noodmunt',
    'City map of Breda in relief': 'Stadsplan',
    'Model of a 24-pound gun': 'saluutkanon',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper Adriaan',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht',
    'Military Operations in flanders with siege and capture of breda': 'Obsidio Bredana',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'List met het Turfschip',
    'Turfschip van Breda in 1590': 'List met het Turfschip',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'Las Lanzas',
    'collectie 17de en 18de eeuws metaal militair tuig': 'halfharnas',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Nassau-Siegen',
    'portret van lodewijk van Nassau': 'Willem Lodewijk',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia van Spanje',
    'Het overlijden van Prins Maurits': 'overlijden van Prins Maurits',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen': 'Furie',
    'Sporen van de Nassaus': 'Nassaus',
}

print(f'\nTesting {len(LABEL_TO_DB)} labels with vector search...\n')
print(f'{"Label":<55} {"Top Result":<45} {"Sim"}')
print('='*110)

correct = 0
wrong = []

for cv_label, db_term in LABEL_TO_DB.items():
    # Use the CV label directly as query with e5 prefix
    prefixed = f'query: {cv_label}'
    embedding = model.encode(prefixed).tolist()
    emb_str = '[' + ','.join(map(str, embedding)) + ']'
    
    cur.execute("""
        SELECT identifier, title, object_type,
               1 - (text_embedding <=> %s::vector) as similarity
        FROM knowledge_base
        WHERE text_embedding IS NOT NULL
        ORDER BY text_embedding <=> %s::vector
        LIMIT 1
    """, (emb_str, emb_str))
    
    r = cur.fetchone()
    if r:
        sim = r[3]
        title = r[1][:43]
        identifier = r[0]
        print(f'{cv_label[:53]:<55} {identifier} | {title:<43} {sim:.3f}')
        if sim > 0.80:
            correct += 1
        else:
            wrong.append((cv_label, r[1], sim))
    else:
        print(f'{cv_label[:53]:<55} NO RESULT')

cur.close()
conn.close()

print(f'\n{"="*110}')
print(f'High confidence (>0.80): {correct}/{len(LABEL_TO_DB)}')
print(f'\nLow confidence (<0.80):')
for l, t, s in wrong:
    print(f'  {s:.3f} | {l[:50]} → {t[:50]}')

Loading e5-small-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded!

Testing 33 labels with vector search...

Label                                                   Top Result                                    Sim
Justinus of Nassau                                      BR00003 | Justinus van Nassau                         0.860
Frederick Henry of Orange-Nassau                        ST01626 | Kaart van het beleg van Breda door Frederik 0.810
Maurice of Oranje-Nassau                                ST04863 | Willem van Oranje                           0.844
Equestrian statue of William of Orange                  SMB000456 | Kaart van het Vorstendom Orange en het Comt 0.819
Money of necessity minted during the siege of breda     ST00431 | Kaart van het beleg van Breda door Spinola  0.841
Noodmunten                                              S03385 | Noodmunt Breda 1 stuiver 1625               0.842
City map of Breda in relief                             GT05064 | Breda. Het Liesbos                          0.843
Model of a 24-pound gun        

In [42]:
from sentence_transformers import SentenceTransformer
import psycopg2

print('Loading e5-small-v2...')
model = SentenceTransformer('intfloat/e5-small-v2')
print('Loaded!')

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Oranje Nassau',
    'Maurice of Oranje-Nassau': 'Maurits',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld',
    'Money of necessity minted during the siege of breda': 'Noodmunt',
    'Noodmunten': 'Noodmunt',
    'City map of Breda in relief': 'Stadsplan',
    'Model of a 24-pound gun': 'saluutkanon',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper Adriaan',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht',
    'Military Operations in flanders with siege and capture of breda': 'Obsidio Bredana',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'List met het Turfschip',
    'Turfschip van Breda in 1590': 'List met het Turfschip',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'Las Lanzas',
    'collectie 17de en 18de eeuws metaal militair tuig': 'halfharnas',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Nassau-Siegen',
    'portret van lodewijk van Nassau': 'Willem Lodewijk',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia van Spanje',
    'Het overlijden van Prins Maurits': 'overlijden van Prins Maurits',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen': 'Furie',
    'Sporen van de Nassaus': 'Nassaus',
}

def hybrid_search(cv_label, db_term, top_k=5):
    """
    Hybrid search: combine vector + ILIKE results using RRF scoring.
    Records appearing in both get boosted to top.
    """
    results = {}  # id -> {title, content, score}
    
    # Stage 1: Vector search (top 10)
    prefixed = f'query: {cv_label}'
    embedding = model.encode(prefixed).tolist()
    emb_str = '[' + ','.join(map(str, embedding)) + ']'
    
    cur.execute("""
        SELECT id, identifier, title, object_type, description, content,
               1 - (text_embedding <=> %s::vector) as similarity
        FROM knowledge_base
        WHERE text_embedding IS NOT NULL
        ORDER BY text_embedding <=> %s::vector
        LIMIT 10
    """, (emb_str, emb_str))
    
    vector_rows = cur.fetchall()
    for rank, r in enumerate(vector_rows):
        rid = r[0]
        rrf_score = 1.0 / (60 + rank + 1)  # RRF k=60
        results[rid] = {
            'identifier': r[1],
            'title': r[2],
            'object_type': r[3],
            'description': r[4],
            'content': r[5],
            'vector_sim': r[6],
            'vector_rank': rank,
            'ilike_rank': None,
            'rrf_score': rrf_score,
            'in_both': False
        }
    
    # Stage 2: ILIKE search (top 10)
    cur.execute("""
        SELECT id, identifier, title, object_type, description, content
        FROM knowledge_base
        WHERE title ILIKE %s OR content ILIKE %s
        LIMIT 10
    """, (f'%{db_term}%', f'%{db_term}%'))
    
    ilike_rows = cur.fetchall()
    for rank, r in enumerate(ilike_rows):
        rid = r[0]
        rrf_score = 1.0 / (60 + rank + 1)
        if rid in results:
            # Appeared in both — boost score
            results[rid]['rrf_score'] += rrf_score
            results[rid]['ilike_rank'] = rank
            results[rid]['in_both'] = True
        else:
            results[rid] = {
                'identifier': r[1],
                'title': r[2],
                'object_type': r[3],
                'description': r[4],
                'content': r[5],
                'vector_sim': None,
                'vector_rank': None,
                'ilike_rank': rank,
                'rrf_score': rrf_score,
                'in_both': False
            }
    
    # Sort by RRF score descending
    sorted_results = sorted(results.values(), key=lambda x: x['rrf_score'], reverse=True)
    return sorted_results[:top_k]

# Test all labels
print(f'\nHybrid search results for all {len(LABEL_TO_DB)} labels:\n')
print(f'{"Label":<50} {"Top Result":<45} {"Both?"}')
print('='*105)

correct = 0
wrong = []

for cv_label, db_term in LABEL_TO_DB.items():
    results = hybrid_search(cv_label, db_term)
    if results:
        top = results[0]
        in_both = '✅ BOTH' if top['in_both'] else ('📊 VEC' if top['vector_rank'] is not None else '📝 ILIKE')
        print(f'{cv_label[:48]:<50} {top["identifier"]} | {top["title"][:40]:<42} {in_both}')
        if top['in_both']:
            correct += 1
        else:
            wrong.append((cv_label, top['title'], in_both))
    else:
        print(f'{cv_label[:48]:<50} NO RESULT')

print(f'\n{"="*105}')
print(f'Found in BOTH vector + ILIKE: {correct}/{len(LABEL_TO_DB)}')
print(f'\nOnly in one source:')
for l, t, src in wrong:
    print(f'  {src} | {l[:50]} → {t[:50]}')

Loading e5-small-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded!

Hybrid search results for all 33 labels:

Label                                              Top Result                                    Both?
Justinus of Nassau                                 ST04852 | Justinus van Nassau                        ✅ BOTH
Frederick Henry of Orange-Nassau                   ST01626 | Kaart van het beleg van Breda door Frede   📊 VEC
Maurice of Oranje-Nassau                           ST04863 | Willem van Oranje                          📊 VEC
Equestrian statue of William of Orange             SMB000456 | Kaart van het Vorstendom Orange en het C   📊 VEC
Money of necessity minted during the siege of br   ST00431 | Kaart van het beleg van Breda door Spino   📊 VEC
Noodmunten                                         S00753 | Noodmunt Breda 20 stuiver 1625             ✅ BOTH
City map of Breda in relief                        GT05064 | Breda. Het Liesbos                         📊 VEC
Model of a 24-pound gun                            ST00969b | Oorlogsscha

In [43]:
print(f'\nTOP 3 HYBRID RESULTS PER LABEL:\n')

for cv_label, db_term in LABEL_TO_DB.items():
    results = hybrid_search(cv_label, db_term, top_k=3)
    print(f'\n{"="*60}')
    print(f'LABEL: {cv_label}')
    print(f'SEARCH TERM: {db_term}')
    for i, r in enumerate(results):
        src = '✅BOTH' if r['in_both'] else ('📊VEC' if r['vector_rank'] is not None else '📝ILIKE')
        sim = f"{r['vector_sim']:.3f}" if r['vector_sim'] else 'N/A'
        print(f'  #{i+1} [{src}] {r["identifier"]} | {r["title"][:50]} | sim={sim}')


TOP 3 HYBRID RESULTS PER LABEL:


LABEL: Justinus of Nassau
SEARCH TERM: Justinus van Nassau
  #1 [✅BOTH] S02764 | Francois de l’Aubespine | sim=0.815
  #2 [✅BOTH] BR00003 | Justinus van Nassau | sim=0.860
  #3 [✅BOTH] ST04854 | Justinus van Nassau | sim=0.828

LABEL: Frederick Henry of Orange-Nassau
SEARCH TERM: Frederik Hendrik van Oranje Nassau
  #1 [📊VEC] ST01626 | Kaart van het beleg van Breda door Frederik Hendri | sim=0.810
  #2 [📝ILIKE] BR00013 | Portret van Frederik Hendrik van Oranje Nassau | sim=N/A
  #3 [📊VEC] ST04673 | Kaart van het vorstendom Orange | sim=0.809

LABEL: Maurice of Oranje-Nassau
SEARCH TERM: Maurits
  #1 [📊VEC] ST04863 | Willem van Oranje | sim=0.844
  #2 [📝ILIKE] ST02492 | Gymnastiekvereniging Pro Patria en Harmonie Consta | sim=N/A
  #3 [📊VEC] ST04859 | Willem van Oranje | sim=0.842

LABEL: Equestrian statue of William of Orange
SEARCH TERM: ruiterstandbeeld
  #1 [📊VEC] SMB000456 | Kaart van het Vorstendom Orange en het Comtat Vena | sim=0.819
  #2 [📝ILI

In [44]:
def hybrid_search_v2(cv_label, db_term, top_k=5):
    results = {}
    
    # Stage 1: Vector search
    prefixed = f'query: {cv_label}'
    embedding = model.encode(prefixed).tolist()
    emb_str = '[' + ','.join(map(str, embedding)) + ']'
    
    cur.execute("""
        SELECT id, identifier, title, object_type, description, content,
               1 - (text_embedding <=> %s::vector) as similarity
        FROM knowledge_base
        WHERE text_embedding IS NOT NULL
        ORDER BY text_embedding <=> %s::vector
        LIMIT 10
    """, (emb_str, emb_str))
    
    for rank, r in enumerate(cur.fetchall()):
        rid = r[0]
        results[rid] = {
            'identifier': r[1], 'title': r[2], 'object_type': r[3],
            'description': r[4], 'content': r[5], 'vector_sim': r[6],
            'vector_rank': rank, 'ilike_rank': None,
            'rrf_score': 1.0 / (60 + rank + 1),
            'in_both': False
        }
    
    # Stage 2: ILIKE search — give 3x boost
    cur.execute("""
        SELECT id, identifier, title, object_type, description, content
        FROM knowledge_base
        WHERE title ILIKE %s OR content ILIKE %s
        LIMIT 10
    """, (f'%{db_term}%', f'%{db_term}%'))
    
    for rank, r in enumerate(cur.fetchall()):
        rid = r[0]
        # ILIKE gets 3x RRF weight — it's more precise
        ilike_rrf = 3.0 / (60 + rank + 1)
        if rid in results:
            results[rid]['rrf_score'] += ilike_rrf
            results[rid]['ilike_rank'] = rank
            results[rid]['in_both'] = True
        else:
            results[rid] = {
                'identifier': r[1], 'title': r[2], 'object_type': r[3],
                'description': r[4], 'content': r[5], 'vector_sim': None,
                'vector_rank': None, 'ilike_rank': rank,
                'rrf_score': ilike_rrf, 'in_both': False
            }
    
    sorted_results = sorted(results.values(), key=lambda x: x['rrf_score'], reverse=True)
    return sorted_results[:top_k]

# Test
print(f'{"Label":<50} {"Top Result":<45} {"Src"}')
print('='*100)

correct = 0
for cv_label, db_term in LABEL_TO_DB.items():
    results = hybrid_search_v2(cv_label, db_term)
    if results:
        top = results[0]
        src = '✅BOTH' if top['in_both'] else ('📊VEC' if top['vector_rank'] is not None else '📝ILIKE')
        print(f'{cv_label[:48]:<50} {top["identifier"]} | {top["title"][:40]:<42} {src}')
        if top['in_both']:
            correct += 1

print(f'\nIn BOTH: {correct}/{len(LABEL_TO_DB)}')

Label                                              Top Result                                    Src
Justinus of Nassau                                 ST04852 | Justinus van Nassau                        ✅BOTH
Frederick Henry of Orange-Nassau                   BR00013 | Portret van Frederik Hendrik van Oranje    📝ILIKE
Maurice of Oranje-Nassau                           GT04074 | Overwinning van Maurits bij Turnhout, 15   📝ILIKE
Equestrian statue of William of Orange             ST04861 | Willem van Oranje                          📝ILIKE
Money of necessity minted during the siege of br   S00746 | Noodmunt Breda 40 stuiver uit 1577         📝ILIKE
Noodmunten                                         S00753 | Noodmunt Breda 20 stuiver 1625             ✅BOTH
City map of Breda in relief                        BR00038 | Stadsplan in reliëf van de stad Breda      📝ILIKE
Model of a 24-pound gun                            S00276 | Klein kanon, saluutkanon                   📝ILIKE
Constantijn Huyg

In [45]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

cur.execute("""
    SELECT 
        MIN(LENGTH(content)) as min_len,
        MAX(LENGTH(content)) as max_len,
        AVG(LENGTH(content)) as avg_len,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY LENGTH(content)) as median_len,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY LENGTH(content)) as p75_len,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY LENGTH(content)) as p90_len,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY LENGTH(content)) as p95_len
    FROM knowledge_base
    WHERE content IS NOT NULL
""")
r = cur.fetchone()
print(f'Min:    {r[0]} chars')
print(f'Max:    {r[1]} chars')
print(f'Avg:    {int(r[2])} chars')
print(f'Median: {int(r[3])} chars')
print(f'P75:    {int(r[4])} chars')
print(f'P90:    {int(r[5])} chars')
print(f'P95:    {int(r[6])} chars')

# Also check our specific museum artifacts
print('\nBR% artifact content lengths:')
cur.execute("""
    SELECT identifier, title, LENGTH(content) as len
    FROM knowledge_base
    WHERE identifier LIKE 'BR%'
    ORDER BY LENGTH(content) DESC
""")
for r in cur.fetchall():
    print(f'  {r[0]} | {r[1][:40]:<40} | {r[2]} chars')

cur.close()
conn.close()

Min:    4 chars
Max:    9393 chars
Avg:    327 chars
Median: 263 chars
P75:    374 chars
P90:    550 chars
P95:    744 chars

BR% artifact content lengths:
  BR00038 | Stadsplan in reliëf van de stad Breda    | 1157 chars
  BR00024 | Fragment van Voetwassing en Laatste Avon | 1016 chars
  BR00014 | Drinkglas in de vorm van een kanon       | 992 chars
  BR00031 | Mogelijk fragment van de Voetwassing uit | 868 chars
  BR00019 | Tazza                                    | 865 chars
  BR00030 | Vernield beeld uit de Grote Kerk         | 800 chars
  BR00029 | Laatste Avondmaal uit de Grote Kerk      | 798 chars
  BR00053 | Portret van Hendrik III van Nassau       | 754 chars
  BR00054 | Portret van Rene van Chalon              | 698 chars
  BR00011 | Portret Willem van Oranje                | 682 chars
  BR00032 | Filips Willem, prins van Oranje          | 664 chars
  BR00003 | Justinus van Nassau                      | 651 chars
  BR00022 | Fragment van Maria met Christuskind      | 651 cha

In [46]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# ACTUAL current LABEL_TO_DB from llm_service.py
CURRENT_LABEL_TO_DB = {
    "Justinus of Nassau": "Justinus van Nassau",
    "Frederick Henry of Orange-Nassau": "Frederik Hendrik",
    "Maurice of Oranje-Nassau": "Maurits",
    "Equestrian statue of William of Orange": "ruiterstandbeeld Willem van Oranje",
    "Money of necessity minted during the siege of breda": "Noodmunten",
    "City map of Breda in relief": "Stadsplan in reliëf",
    "Model of a 24-pound gun": "kanon",
    "Constantijn Huygens, curator van de Illustere School in Breda": "Constantijn Huygens",
    "Beeld vna de turfschipper": "turfschipper",
    "Departure of the Spanish occupation from Breda on 10 October 1637": "Uittocht van de Spaanse bezetting",
    "Military Operations in flanders with siege and capture of breda": "beleg van Breda",
    "Retreat of the Spanish Garrison from Breda": "Uittocht van de Spaanse bezetting",
    "Obsidio Bredana": "Obsidio Bredana",
    "het beleg van Breda": "beleg van Breda",
}

def current_search(query, limit=5):
    # Stage 1: FTS simple
    cur.execute("""
        SELECT identifier, title, object_type
        FROM knowledge_base
        WHERE to_tsvector('simple',
            coalesce(title,'') || ' ' ||
            coalesce(creator,'') || ' ' ||
            coalesce(description,'') || ' ' ||
            coalesce(content,'')
        ) @@ plainto_tsquery('simple', %s)
        ORDER BY ts_rank(
            to_tsvector('simple', coalesce(title,'') || ' ' || coalesce(creator,'') || ' ' || coalesce(description,'') || ' ' || coalesce(content,'')),
            plainto_tsquery('simple', %s)
        ) DESC
        LIMIT %s
    """, (query, query, limit))
    rows = cur.fetchall()
    if rows:
        return 'FTS', rows

    # Stage 2: ILIKE fallback
    keywords = [w for w in query.split() if len(w) >= 3]
    if not keywords:
        return 'NONE', []
    conditions = ' OR '.join(['(title ILIKE %s OR creator ILIKE %s OR description ILIKE %s OR content ILIKE %s)' for _ in keywords])
    params = []
    for kw in keywords:
        params.extend([f'%{kw}%'] * 4)
    params.append(limit)
    cur.execute(f"SELECT identifier, title, object_type FROM knowledge_base WHERE {conditions} LIMIT %s", params)
    rows = cur.fetchall()
    if rows:
        return 'ILIKE', rows
    return 'NONE', []

print('CURRENT RETRIEVAL (actual llm_service.py LABEL_TO_DB)\n')
print(f'{"CV Label":<50} {"DB Term":<35} {"Method":<6} {"Top Result"}')
print('='*130)

for cv_label, db_term in CURRENT_LABEL_TO_DB.items():
    method, rows = current_search(db_term)
    top = rows[0] if rows else None
    result_str = f'{top[0]} | {top[1][:40]}' if top else 'NO RESULT'
    print(f'{cv_label[:48]:<50} {db_term[:33]:<35} {method:<6} {result_str}')

cur.close()
conn.close()

CURRENT RETRIEVAL (actual llm_service.py LABEL_TO_DB)

CV Label                                           DB Term                             Method Top Result
Justinus of Nassau                                 Justinus van Nassau                 FTS    ST04854 | Justinus van Nassau
Frederick Henry of Orange-Nassau                   Frederik Hendrik                    FTS    SMB000533 | Portret van Frederik Hendrik
Maurice of Oranje-Nassau                           Maurits                             FTS    BR00012 | Maurits, prins van Oranje-Nassau
Equestrian statue of William of Orange             ruiterstandbeeld Willem van Oranj   FTS    SMB000416 | Ruiterstandbeeld van Willem van Oranje
Money of necessity minted during the siege of br   Noodmunten                          FTS    S00765 | Noodmunt Breda 20 stuiver 1577
City map of Breda in relief                        Stadsplan in reliëf                 FTS    BR00038 | Stadsplan in reliëf van de stad Breda
Model of a 24-pound gun

In [47]:
from sentence_transformers import SentenceTransformer
import psycopg2

model = SentenceTransformer('intfloat/e5-small-v2')
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Real visitor questions someone might ask in the museum
visitor_questions = [
    # English questions
    "Who was Ambrogio Spinola and why is he important for Breda?",
    "Tell me about the siege of Breda",
    "What happened to Justinus van Nassau?",
    "How did Maurits capture Breda using the peat boat?",
    "What are the emergency coins from the siege?",
    "Who was Frederick Henry of Orange?",
    "Tell me about the city map of Breda",
    "What is the Obsidio Bredana book about?",
    "Who was Isabella Clara Eugenia?",
    "What is the partizaan weapon?",
    # Dutch questions
    "Wie was Ambrogio Spinola?",
    "Vertel me over het beleg van Breda",
    "Wat is de overgave van Breda?",
    "Wie was Justinus van Nassau?",
]

print('CONVERSATIONAL QUESTION ANALYSIS\n')
print('Testing if vector search finds relevant content for real visitor questions\n')

for question in visitor_questions:
    prefixed = f'query: {question}'
    embedding = model.encode(prefixed).tolist()
    emb_str = '[' + ','.join(map(str, embedding)) + ']'
    
    cur.execute("""
        SELECT identifier, title, object_type,
               LEFT(content, 300) as content_preview,
               1 - (text_embedding <=> %s::vector) as similarity
        FROM knowledge_base
        WHERE text_embedding IS NOT NULL
        ORDER BY text_embedding <=> %s::vector
        LIMIT 3
    """, (emb_str, emb_str))
    
    rows = cur.fetchall()
    print(f'\n{"="*70}')
    print(f'Q: "{question}"')
    for r in rows:
        print(f'\n  [{r[4]:.3f}] {r[0]} | {r[1]} | {r[2]}')
        print(f'  {r[3][:200] if r[3] else "NULL"}')

cur.close()
conn.close()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CONVERSATIONAL QUESTION ANALYSIS

Testing if vector search finds relevant content for real visitor questions


Q: "Who was Ambrogio Spinola and why is he important for Breda?"

  [0.872] ST00588 | Ambrogio Spinola | Prent
  Ambrogio Spinola Ambrogio Spinola is de topgeneraal aan Spaanse zijde. Onder zijn leiding herovert het Spaanse leger Breda in 1625 op de Oranje-Nassaus. Omdat de verdediging te sterk is voor een gewel

  [0.856] ST00430 | Kaart van het beleg van Breda door Spinola | Prent
  Kaart van het beleg van Breda door Spinola Abraham Verhoeven Volledige titel: Oprechte ende waerachtige Af-beeldinge van het Belegh van de Stercke Stadt Breda, die belegert is geworden den 27. Augusti

  [0.854] ST01524 | Kaart van het beleg van Breda door Spinola | Prent
  Kaart van het beleg van Breda door Spinola J. van Heijden Volledige titel: Breda ab Excellentissimo March. Ambr. Spinola nomine Catholici Regis obsessa cum oppositis illustriss. Mauritii Nassauii cast

Q: "Tell me about the si

In [49]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Check actual columns
print('=== brabant_europeana COLUMNS ===')
cur.execute("""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_name = 'brabant_europeana' 
    ORDER BY ordinal_position
""")
for r in cur.fetchall():
    print(f'  {r[0]:<30} {r[1]}')

# Sample record
print('\n=== SAMPLE RECORD ===')
cur.execute("SELECT * FROM brabant_europeana LIMIT 1")
cols = [d[0] for d in cur.description]
row = cur.fetchone()
for col, val in zip(cols, row):
    if val:
        print(f'  {col:<25} {str(val)[:150]}')

# Breda content count
print('\n=== BREDA CONTENT ===')
cur.execute("SELECT COUNT(*) FROM brabant_europeana WHERE title ILIKE '%Breda%'")
print(f'  Breda in title: {cur.fetchone()[0]}')

cur.close()
conn.close()

=== brabant_europeana COLUMNS ===
  id                             bigint
  source                         text
  content                        text
  embedding                      USER-DEFINED
  tsv                            tsvector
  title                          text
  object_type                    text
  provider                       text
  country                        text
  year                           text
  place                          text
  url                            text
  latitude                       numeric
  longitude                      numeric

=== SAMPLE RECORD ===
  id                        69390
  source                    brabant_cloud
  content                   Sint-Antonius van Paduakerk
Sint Antonius van Paduakerk
Heilige Antonius van Paduakerk
Antoniuskerk
St. Antonius van Paduakerk
H. Antonius van Paduake
  embedding                 [-0.10004896,0.027584255,0.017473916,0.013089399,-0.058923356,-0.013833912,0.03439355,-0.028790418,0.0337484

In [50]:
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

# Check embedding dimensions
cur.execute("SELECT vector_dims(embedding) FROM brabant_europeana WHERE embedding IS NOT NULL LIMIT 1")
r = cur.fetchone()
print(f'Embedding dimensions: {r[0]}')

# Sample Breda records
print('\n=== SAMPLE BREDA RECORDS ===')
cur.execute("""
    SELECT id, title, object_type, year, place, LEFT(content, 200)
    FROM brabant_europeana 
    WHERE title ILIKE '%Breda%'
    LIMIT 10
""")
for r in cur.fetchall():
    print(f'\n  {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]}')
    print(f'  {r[5]}')

# Check object types for Breda content
print('\n=== BREDA OBJECT TYPES ===')
cur.execute("""
    SELECT object_type, COUNT(*) as count
    FROM brabant_europeana 
    WHERE title ILIKE '%Breda%'
    GROUP BY object_type
    ORDER BY count DESC
    LIMIT 15
""")
for r in cur.fetchall():
    print(f'  {r[0]:<30} {r[1]}')

cur.close()
conn.close()

Embedding dimensions: 384

=== SAMPLE BREDA RECORDS ===

  70322 | Klooster van de Catechisten van Breda | Kloostergebouw | 1885 | Etten-Leur
  Klooster van de Catechisten van Breda
Markt 60
Kloostergebouw
Rijksmonument
1885
1950
Bestaand
Niet meer in gebruik
Piet van Genk
Petrus Johannes van Genk
P.J. van Genk
Rooms-Katholieke Kerk
Rooms Kat

  70655 | Klooster aan de Bredalaan | Kloostergebouw | 2e millennium | Eindhoven
  Klooster aan de Bredalaan
Kloostergebouw
2e millennium
1977
Bestaand
Niet meer in gebruik
Rooms-Katholieke Kerk
Katholieke Kerk
Rooms Katholiek
Rooms-Katholiek
Bredalaan 50, Eindhoven
Bredalaan 50
Ein

  71890 | Bowlingwedstrijd Breda - Antwerpen | VIDEO |  | 
  Bowlingwedstrijd Breda - Antwerpen | Netherlands Institute for Sound & Vision | Netherlands | VIDEO | coordinates: 51.58889,4.775833 | http://creativecommons.org/licenses/by-sa/4.0/ | https://www.euro

  71891 | Transport of a fractionation column from Breda to Pernis | VIDEO |  | 
  Transport of a fraction

In [52]:
from sentence_transformers import SentenceTransformer
import psycopg2

model = SentenceTransformer('intfloat/e5-small-v2')
conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

questions = [
    'Who was Ambrogio Spinola and why is he important for Breda?',
    'Tell me about the siege of Breda 1625',
    'What is the Obsidio Bredana book?',
    'How did Maurits capture Breda using the peat boat in 1590?',
    'Who was Justinus van Nassau governor of Breda?',
]

for question in questions:
    embedding = model.encode(f'query: {question}').tolist()
    emb_str = '[' + ','.join(map(str, embedding)) + ']'
    
    cur.execute("""
        SELECT id, title, object_type, year,
               LEFT(content, 200) as preview,
               1 - (embedding <=> %s::vector) as similarity
        FROM brabant_europeana
        ORDER BY embedding <=> %s::vector
        LIMIT 3
    """, (emb_str, emb_str))
    
    rows = cur.fetchall()
    print(f'\n{"="*65}')
    print(f'Q: "{question}"')
    for r in rows:
        print(f'\n  [{r[5]:.3f}] {r[0]} | {r[1]} | {r[2]} | {r[3]}')
        print(f'  {r[4]}')

cur.close()
conn.close()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Q: "Who was Ambrogio Spinola and why is he important for Breda?"

  [0.832] 106219 | Bootjes in het Spanjaardsgat tijdens evenement Breda Drijft Gezien vanaf de Hoge Brug op de achtergrond links de Granaattoren en rechts de Duiventoren op het terrein van de
          Koninklijke Militaire Academie (KMA). Uiterst rechts het horecaschip Spinola | IMAGE | 
  Bootjes in het Spanjaardsgat tijdens evenement Breda Drijft Gezien vanaf de Hoge Brug op de achtergrond links de Granaattoren en rechts de Duiventoren op het terrein van de
          Koninklijke Milit

  [0.832] 89141 | Bootjes in het Spanjaardsgat tijdens evenement Breda Drijft Gezien vanaf de Hoge Brug op de achtergrond links de Granaattoren en rechts de Duiventoren op het terrein van de
          Koninklijke Militaire Academie (KMA). Uiterst rechts het horecaschip Spinola | IMAGE | 
  Bootjes in het Spanjaardsgat tijdens evenement Breda Drijft Gezien vanaf de Hoge Brug op de achtergrond links de Granaattoren en rechts de Duiventor

## NEW VECTOR SEARCH


In [53]:
import requests
import json

BACKEND = 'http://194.171.191.226:3414'

def ask(message, landmark=None, conversation_id='test1'):
    payload = {
        'message': message,
        'conversation_id': conversation_id,
        'audience': 'tourist'
    }
    if landmark:
        payload['landmark'] = landmark
    
    parts = []
    with requests.post(
        f'{BACKEND}/api/chat',
        json=payload,
        stream=True,
        timeout=60
    ) as resp:
        for line in resp.iter_lines():
            if line:
                try:
                    d = json.loads(line.decode('utf-8') if isinstance(line, bytes) else line)
                    if d.get('type') == 'answer':
                        parts.append(d.get('content', ''))
                except:
                    pass
    return ''.join(parts).strip()

# Test all the previously failing labels
tests = [
    ('Equestrian statue of William of Orange', 'Equestrian statue of William of Orange'),
    ('Model of a 24-pound gun', 'Model of a 24-pound gun'),
    ('Constantijn Huygens', None),
    ('Who was Ambrogio Spinola?', None),
    ('Tell me about the siege of Breda', None),
    ('Sporen van de Nassaus', 'Sporen van de Nassaus'),
    ('collectie 17de en 18de eeuws metaal militair tuig', 'collectie 17de en 18de eeuws metaal militair tuig'),
    ('Het overlijden van Prins Maurits', 'Het overlijden van Prins Maurits'),
]

for message, landmark in tests:
    print(f'\n{"="*60}')
    print(f'Q: {message}')
    response = ask(message, landmark)
    print(f'A: {response}')


Q: Equestrian statue of William of Orange
A: The equestrian statue of Willem III of Orange-Nassau stands in Breda's Kasteelplein. It was created by A. Schreurs and placed there in 1921. Earlier designs by Toon Dupuis were considered but not built. The statue shows Willem III, who ruled as stadholder and king. A teacup with the statue's image was made by GERO. The original model was part of a competition in 1903.

Q: Model of a 24-pound gun
A: The Stedelijk Museum Breda holds a model of a 24-pound cannon. It was made by Johan Swijs in 1697. The cannon is decorated with Bredas coat of arms and ornamental details. It represents the guns used in Bredas fortifications. Two similar cannons are displayed in Bredas city hall. They stand near a copy of Diego Velázquezs painting "The Surrender of Breda."

Q: Constantijn Huygens
A: The Stedelijk Museum Breda has a gravure of Constantijn Huygens from 1657. He led the Illustere School founded in 1646 by Prince Frederik Hendrik. The gravure shows h

In [54]:
tests = [
    ('Who was Ambrogio Spinola?', None),
    ('Tell me about the siege of Breda', None),
    ('Tell me about Justinus', None),
    ('Equestrian statue of William of Orange', 'Equestrian statue of William of Orange'),
    ('Model of a 24-pound gun', 'Model of a 24-pound gun'),
    ('Ambrogio Spinola', 'Ambrogio Spinola'),
]

for message, landmark in tests:
    print(f'\nQ: {message}')
    response = ask(message, landmark)
    print(f'A: {response}')


Q: Who was Ambrogio Spinola?
A: Ambrogio Spinola was a top Spanish general during the Eighty Years' War. He led the 1625 recapture of Breda from the Dutch. Spinola avoided a violent siege by cutting off the city's supplies. This forced the surrender of Breda's defenders. His tactics are documented in museum collections. Spinola's role is highlighted in historical prints and coins.

Q: Tell me about the siege of Breda
A: The siege of Breda in 1624-25 is shown on maps by J. van Bergen and others. These maps detail Spanish positions and the failed relief attempt by Prince Maurice. One map by Jacques Callot shows the entire siege in six panels. Another by Jan van Hilten focuses on Frederick Henry's siege in 1637. Abraham Verhoeven's map highlights Spanish perspectives and Jesuit chapels. These maps document military strategies and historical events during the conflict.

Q: Tell me about Justinus
A: Justinus van Nassau was the second son of William of Orange. He never became a prince of Or

In [55]:
tests = [
    ('Who was Ambrogio Spinola?', None),
    ('Tell me about the siege of Breda', None),
    ('What are the emergency coins from the siege?', None),
    ('Tell me about Justinus', None),
    ('Equestrian statue of William of Orange', 'Equestrian statue of William of Orange'),
    ('Model of a 24-pound gun', 'Model of a 24-pound gun'),
]

for message, landmark in tests:
    print(f'\nQ: {message}')
    response = ask(message, landmark)
    print(f'A: {response}')


Q: Who was Ambrogio Spinola?
A: Ambrogio Spinola was a Spanish general who captured Breda in 1625. He avoided a violent siege by blockading the city. This led to a food shortage forcing Justinus van Nassau to surrender. A portrait of him shows his golden fleece decorations. Artists include Anthony van Dyck and Pieter Soutman. A medal celebrates Breda's surrender to his forces.

Q: Tell me about the siege of Breda
A: The siege of Breda in 16241625 is shown in maps by J. van Bergen, le Grain, and Jacques Callot. These maps detail Spanish forces under Spinola and Dutch efforts led by Prince Maurits. One map shows the failed relief attempt by Prince Maurits in October 1624. Another by Jan van Hilten depicts the siege by Frederick Hendrik in 1637. The Spanish perspective is captured in Abraham Verhoevens map, including Jesuit services for soldiers. These documents highlight military strategies and historical events.

Q: What are the emergency coins from the siege?
A: No information about e

In [56]:
tests = [
    ('Noodmunten', 'Noodmunten'),
    ('Vertel me over de noodmunten', None),
    ('Wat zijn de noodmunten?', None),
    ('noodmunten breda', None),
]

for message, landmark in tests:
    print(f'\nQ: {message}')
    response = ask(message, landmark)
    print(f'A: {response}')


Q: Noodmunten
A: No information about Noodmunten is available in the provided knowledge base. The entries focus on maps, paintings, and historical figures related to Breda's history. None mention Noodmunten or emergency coins. The museum's collection includes artworks but not numismatic items from this period.

Q: Vertel me over de noodmunten
A: Breda's 1625 siege led to emergency coins. A 60 stuiver silver coin was minted but later found to be a replica. A 2 stuiver coin bears "Breda Obsessa 1625" and a shield. A 1 stuiver copper coin was also made. These coins used melted citizen metals. They paid soldiers during the Spanish siege.

Q: Wat zijn de noodmunten?
A: Noodmunten van Breda zijn historische muntstukken uit 1625. Ze werden gemaakt van zilver of koper en hadden waarden zoals 40 of 1 stuiver. Een 60 stuiver munt is waarschijnlijk een replica. De stempels bevatten heraldische afbeeldingen. Ze staan vermeld in het Stedelijk Museum Breda. De munten dateren van de 17e eeuw.

Q: no

In [57]:
import psycopg2

conn = psycopg2.connect(host='194.171.191.227', port=5432, dbname='o2', user='postgres', password='postgres')
cur = conn.cursor()

def show_full_artifact(search_term):
    """Show complete DB record for any artifact."""
    cur.execute("""
        SELECT identifier, title, creator, description, object_type, 
               subject, medium, extent, technique, collection,
               date_created, place, content, url_is_shown_at, url_thumb_large
        FROM knowledge_base 
        WHERE title ILIKE %s OR content ILIKE %s
        ORDER BY 
            CASE WHEN identifier LIKE 'BR%%' THEN 0
                 WHEN identifier LIKE 'S%%' THEN 1
                 ELSE 2 END
        LIMIT 3
    """, (f'%{search_term}%', f'%{search_term}%'))
    
    rows = cur.fetchall()
    cols = ['identifier','title','creator','description','object_type',
            'subject','medium','extent','technique','collection',
            'date_created','place','content','url_is_shown_at','url_thumb_large']
    
    print(f'\n{"="*70}')
    print(f'SEARCH: "{search_term}" → {len(rows)} results')
    print('='*70)
    
    for i, row in enumerate(rows, 1):
        print(f'\n--- Record {i} ---')
        for col, val in zip(cols, row):
            if val and not col.endswith('embedding'):
                print(f'  {col:<20} {str(val)[:200]}')

# Test with different artifacts
artifacts_to_check = [
    'Justinus van Nassau',
    'Ambrogio Spinola', 
    'saluutkanon',
    'Noodmunt',
    'turfschipper',
    'Herdenkingsbeker',
    'Obsidio Bredana',
    'Las Lanzas',
]

for term in artifacts_to_check:
    show_full_artifact(term)

cur.close()
conn.close()


SEARCH: "Justinus van Nassau" → 3 results

--- Record 1 ---
  identifier           BR00003
  title                Justinus van Nassau
  creator              anoniem, Jan Anthonisz. Ravesteyn
  description          Justinus is de tweede zoon van Willem van Oranje. Hij draagt de naam 'Van Nassau' maar zal geen prins van Oranje worden. Zijn vader en moeder waren niet getrouwd, wat hem een 'onechte zoon' maakte. Al
  object_type          Schilderij
  subject              Portret, Persoon, Man
  medium               Paneel, Hout, Olieverf
  extent               hoogte: 36 cm, breedte: 28.6 cm
  collection           Stedelijk Museum Breda
  date_created         1607 – 1700
  content              Justinus van Nassau anoniem, Jan Anthonisz. Ravesteyn Justinus is de tweede zoon van Willem van Oranje. Hij draagt de naam 'Van Nassau' maar zal geen prins van Oranje worden. Zijn vader en moeder ware
  url_is_shown_at      https://www.stedelijkmuseumbreda.nl/nl/collectie?diw-id=brabantcloud_stedeli